# JADCO — prévision de la croissance des loyers effectifs en 2026

Copie de travail adaptée du notebook de départ officiel. Ce notebook contient les transformations et méthodes d'analyse, mais aucune donnée d'entrée ni sortie enregistrée. Les fichiers du défi doivent être fournis séparément depuis un emplacement local autorisé.

**Question :** quelle médiane de hausse annualisée du loyer effectif peut-on estimer pour 2026 parmi les transitions comparables?

La copie publique ne contient pas les résultats calculés sur les données CRM. L'estimation et le backtest s'affichent uniquement lorsqu'une personne exécute le notebook avec ses propres fichiers autorisés.

## A — Définition et hypothèses

La cible est la **médiane des variations annualisées de `sRentEffective`**, entre baux consécutifs d'une même unité, dont le nouveau bail commence pendant l'année cible. Clé : `sPropCode + sUnitCode`; chaque transition valide a un poids égal. Une unité peut contribuer plusieurs transitions; la dernière transition par unité/an est une sensibilité distincte.

Annualisation : `100 * ((prix / prix_précédent)**(1 / écart_en_années) - 1)`, avec 365,25 jours/an et écarts strictement entre 0,5 et 2,5 ans, selon le starter. L'année correspond au début de bail; la signature sert à filtrer la disponibilité.

La population exclut les premiers baux et les transitions non comparables. Les poids des immeubles varient donc avec les paires disponibles. La médiane globale est calculée sur les transitions réunies, jamais comme une moyenne des médianes d'immeubles. L'extrait ne fournit ni occupation, ni absorption, ni recettes totales de l'entreprise.

**Disponibilité :** scénario principal contractuel, supposant les loyers effectifs fournis connus pour les contrats signés/commencés à la coupure. Une sensibilité conservatrice limite l'historique aux baux terminés avant la coupure. La date de saisie/correction n'est pas fournie : ces hypothèses ne prouvent pas une reconstruction historique parfaite. Les instantanés financiers du référentiel 2025 ne sont pas des prédicteurs des années précédentes.

In [ ]:
from pathlib import Path
import os, io, json, zipfile, hashlib, platform, inspect
from dataclasses import dataclass, replace, asdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path.cwd()
PUBLIC_DIR = ROOT / 'externes'
PUBLIC_DIR.mkdir(exist_ok=True)
TABLE_FILES = {name: f'equinoxe_{name}.csv' for name in
               ['listings', 'lease_history', 'concessions', 'asking_history']}
INPUT_DIR = Path(os.environ.get('JADCO_INPUT_DIR', str(ROOT / 'entrees-locales')))
PACKAGE_PATH = Path(os.environ.get('JADCO_PACKAGE_PATH', str(ROOT.parent / 'jadco-participants.zip')))
PUBLIC_ARCHIVE = ROOT.parent / 'externes-local' / 'statcan-18100004-en.zip'
PUBLIC_METADATA = ROOT.parent / 'externes-local' / 'statcan-18100004-metadata.json'

@dataclass(frozen=True)
class Protocol:
    min_gap_years: float = 0.5
    max_gap_years: float = 2.5
    days_per_year: float = 365.25
    history_years: int = 3
    min_segment_pairs: int = 30
    min_output_pairs: int = 30
    cpi_anchor_weight: float = 0.5
    complexity_gain_pp: float = 0.25
    availability: str = 'contractual'
    last_pair_per_unit: bool = False
    clip_quantiles: tuple = (0.0, 1.0)

CONFIG = Protocol()
UNIT_KEY = ['sPropCode', 'sUnitCode']
TEST_YEARS = (2023, 2024, 2025)
METHODS = ('A_derniere', 'B_robuste_segmentee', 'C_B_plus_IPC')
SELECTED_METHOD = 'A_derniere'
COLORS = {'contractual': '#245780', 'effective': '#a16b18', 'naive': '#676767'}
plt.rcParams.update({'figure.figsize': (9, 4.5), 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.alpha': 0.2})
pd.set_option('display.max_rows', 24)
pd.set_option('display.max_columns', 9)
print('Python', platform.python_version(), '| pandas', pd.__version__, '| NumPy', np.__version__)

## B — Charger et contrôler les sources

Les entrées restent hors du dossier de remise. Fournir les quatre CSV dans `JADCO_INPUT_DIR`, ou désigner le ZIP officiel par `JADCO_PACKAGE_PATH`. Aucun téléchargement réseau n'est effectué pendant l'exécution. Les zéros initiaux des numéros d'unité sont préservés. Les sorties ci-dessous ne contiennent aucune ligne par unité.

In [ ]:
def load_crm():
    if all((INPUT_DIR / name).is_file() for name in TABLE_FILES.values()):
        frames = {k: pd.read_csv(INPUT_DIR / f, dtype={'sUnitCode': 'string'}) for k, f in TABLE_FILES.items()}
        integrity = 'CSV locaux; manifeste ZIP non utilisé'
    elif PACKAGE_PATH.is_file():
        with zipfile.ZipFile(PACKAGE_PATH) as archive:
            manifest = json.loads(archive.read('manifest.json'))
            assert all(hashlib.sha256(archive.read(f['path'])).hexdigest() == f['sha256']
                       and len(archive.read(f['path'])) == f['bytes'] for f in manifest['files']), 'Intégrité du package incorrecte'
            frames = {k: pd.read_csv(io.BytesIO(archive.read(f)), dtype={'sUnitCode': 'string'}) for k, f in TABLE_FILES.items()}
        integrity = 'Tailles et SHA-256 du manifeste conformes'
    else:
        raise FileNotFoundError('Fournir les quatre CSV localement ou configurer JADCO_PACKAGE_PATH.')
    return frames, integrity

tables, integrity = load_crm()
listings, leases_raw, concessions_raw, asking_raw = [tables[k] for k in TABLE_FILES]
quality = pd.DataFrame([{'table': k, 'lignes': len(v), 'colonnes': v.shape[1],
                        'cellules_manquantes': int(v.isna().sum().sum()),
                        'doublons_complets': int(v.duplicated().sum()),
                        'unites_distinctes': int(v.hUnit.nunique())} for k,v in tables.items()])
display(quality)
print(integrity)

### Dates, clés et dictionnaire de lecture

`h` signifie handle/identifiant; `s` signifie valeur stockée, pas nécessairement texte. `sRent` est contractuel; `sRentEffective` contient déjà l'ajustement économique de concessions. `sRenewal=1` correspond au renouvellement et `0` à la relocation. `sTermMonths` est la durée du bail, alors que l'annualisation compare deux dates de début. `sDateFrom/To` sont des dates d'application du crédit, pas sa date de décision.

Les descriptions des charges ne suffisent pas à expliquer leur périodicité. Les caractéristiques structurelles sont actuellement cohérentes dans les tables, mais cette cohérence ne remplace pas des instantanés historiques.

In [ ]:
def prepare_leases(frame):
    result = frame.copy()
    for col in ['sLeaseFrom', 'sLeaseTo', 'sSignDate', 'sAvailable']:
        result[col] = pd.to_datetime(result[col], errors='raise')
    result['year'] = result.sLeaseFrom.dt.year
    return result

leases = prepare_leases(leases_raw)
concessions = concessions_raw.copy()
for col in ['sDateFrom', 'sDateTo']:
    concessions[col] = pd.to_datetime(concessions[col], errors='raise')
asking = asking_raw.copy()
asking['date'] = pd.to_datetime(asking.sMonth + '-01')
asking['month_end'] = asking.date + pd.offsets.MonthEnd(0)
assert not listings.duplicated(UNIT_KEY).any()
assert not listings.hUnit.duplicated().any()
assert not leases.duplicated(UNIT_KEY + ['sLeaseFrom']).any()
assert not leases.duplicated(['hUnit', 'sTermSeq']).any()
assert leases.sRent.gt(0).all() and leases.sRentEffective.gt(0).all()
assert leases.sRenewal.isin([0,1]).all()
assert leases.sConcession.isin([0,1]).all()
assert leases.sLeaseTo.ge(leases.sLeaseFrom).all()
assert leases.sSignDate.le(leases.sLeaseFrom).all()
assert not asking.duplicated(['hUnit','sMonth']).any()
safe_join = leases.merge(listings[['hUnit']], on='hUnit', validate='many_to_one')
assert len(safe_join) == len(leases)
ordered = leases.sort_values(UNIT_KEY + ['sLeaseFrom'])
prior_end = ordered.groupby(UNIT_KEY).sLeaseTo.shift()
key_checks = {'unites_collision_cle_interdite': int(listings.duplicated(['sSite','sUnitCode'], keep=False).sum()),
              'baux_avant_jointure': len(leases), 'baux_apres_jointure': len(safe_join),
              'unites_baux_inconnues': int((~leases.hUnit.isin(listings.hUnit)).sum()),
              'chevauchements_baux': int((ordered.sLeaseFrom <= prior_end).sum())}
display(pd.Series(key_checks, name='compte').to_frame())
date_coverage = pd.DataFrame([{'champ': col, 'minimum': str(leases[col].min().date()),
                              'maximum': str(leases[col].max().date()),
                              'apres_2025': int((leases[col] > pd.Timestamp('2025-12-31')).sum())}
                             for col in ['sSignDate','sLeaseFrom','sLeaseTo']])
display(date_coverage)

## C — Composition : ce que la médiane brute masque

Les statistiques annuelles ci-dessous utilisent tous les baux débutant cette année, puis la comparaison à unité constante utilise une population plus restreinte. L'arrivée d'immeubles plus chers et le changement de typologies peuvent déplacer la médiane sans refléter la hausse d'une même unité. Le premier bail observé ne prouve pas la date juridique de première occupation.

In [ ]:
annual = leases.groupby('year').agg(baux=('hUnit','size'), loyer_contractuel_median=('sRent','median'),
               loyer_effectif_median=('sRentEffective','median'), superficie_mediane=('sSqft','median'),
               part_deux_chambres_plus=('sBeds', lambda x: x.ge(2).mean()), immeubles=('sBuilding','nunique'))
annual['hausse_brute_pct'] = annual.loyer_contractuel_median.pct_change()*100
display(annual.round(2))
mix_counts = leases.groupby(['year','sBuilding']).size().unstack(fill_value=0)
fig, axes = plt.subplots(1,2,figsize=(12,4.5))
annual.superficie_mediane.plot(ax=axes[0], color=COLORS['contractual'], marker='o')
axes[0].set(title='Composition des baux observés', xlabel='Année de début', ylabel='Superficie médiane (pi²)')
(annual.part_deux_chambres_plus*100).plot(ax=axes[1], color=COLORS['effective'], marker='s')
axes[1].set(xlabel='Année de début', ylabel='Baux de 2 chambres et plus (%)', ylim=(0,100))
fig.tight_layout(); plt.show()

## C bis — Quantifier la composition avec la même statistique

Nous conservons la médiane du loyer mensuel de tous les baux débutant dans chaque année. Les strates sont les croisements immeuble/chambres présents dans les deux années. Nous comparons 2022→2023 et 2023→2024. Le support commun signifie mêmes strates, pas mêmes unités.

Sur ce support, Mab désigne la médiane de la distribution des prix de l'année a, repondérée avec les parts de baux par strate de l'année b. Nous combinons les distributions complètes; nous ne moyennons pas les médianes de bâtiments. Les poids sont normalisés, et la médiane pondérée utilise le milieu de l'intervalle médian quand la masse cumulée atteint 0,5 : elle restitue ainsi la médiane ordinaire avec des poids uniformes.

- Distribution dans les strates : [(M10−M00)+(M11−M01)]/2.
- Composition des strates communes : [(M01−M00)+(M11−M10)]/2.
- Extension du support : (médiane totale finale−M11)−(médiane totale initiale−M00).

Ces trois composantes s'additionnent exactement à la différence des médianes brutes en dollars. Les deux premières moyennent les deux ordres possibles de changement des prix et des poids : décomposition symétrique de type Shapley. Les contributions en points divisent chaque composante par la même médiane brute initiale.

**Portée descriptive.** Le terme distribution comprend encore des différences de superficie, étage, unité, concession ou type de bail au sein d'une strate. Il ne mesure pas l'inflation d'un même logement. L'extension mesure un contraste d'inclusion dans cet échantillon, jamais l'effet causal d'une construction. Une sensibilité « immeuble seulement » montre que le choix des strates redistribue les deux premières composantes. Les petites strates restent utilisées localement sans publier leurs valeurs individuelles ni leur attribuer une précision statistique.

Le dernier tableau utilise exactement les mêmes unités dans les deux années, à raison du dernier bail commencé par unité/an. Il distingue variation des niveaux médians et médiane des variations individuelles non annualisées. Ce panel sélectionné n'est pas la cible principale : les deux baux annuels peuvent ne pas être consécutifs. La cible principale reste calculée ensuite à partir des prédécesseurs chronologiques et des durées exactes.


In [ ]:
MIX_YEARS = ((2022, 2023), (2023, 2024))
MIX_STRATA = ("sBuilding", "sBeds")
MIX_UNIT_KEY = ("sPropCode", "sUnitCode")
MIX_PRICE_COLS = ("sRent", "sRentEffective")
MIX_NUMERICAL_ATOL = 1e-9


def mediane_ponderee_intervalle(values, weights):
    """Médiane de la CDF pondérée; milieu de l'intervalle si F atteint 0,5.

    Cette convention restitue np.median avec des poids uniformes, y compris
    lorsque l'effectif est pair. Ce n'est pas une interpolation de quantiles
    déterminée par un effectif artificiel après repondération.
    """
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    if values.size == 0 or values.size != weights.size:
        raise ValueError("Valeurs et poids doivent être non vides et de même taille")
    if not np.isfinite(values).all() or not np.isfinite(weights).all():
        raise ValueError("Valeur/poids non fini")
    if (weights <= 0).any():
        raise ValueError("Poids non strictement positif")
    order = np.argsort(values, kind="stable")
    values, weights = values[order], weights[order]
    cdf = np.cumsum(weights) / weights.sum()
    boundaries = np.flatnonzero(np.isclose(cdf, 0.5, rtol=0, atol=1e-12))
    if boundaries.size:
        index = int(boundaries[0])
        if index + 1 < len(values):
            return float((values[index] + values[index + 1]) / 2)
    return float(values[np.searchsorted(cdf, 0.5)])


def mediane_contrefactuelle(frame, cell_mass, price_col, strata):
    """Distribution de prix du frame, poids de strates donnés par cell_mass."""
    counts = frame.groupby(list(strata), observed=True).size()
    cell_keys = frame[list(strata)].itertuples(index=False, name=None)
    if len(strata) == 1:
        keys = [key[0] for key in cell_keys]
    else:
        keys = list(cell_keys)
    weights = np.array([cell_mass.loc[key] / counts.loc[key] for key in keys])
    assert np.isclose(weights.sum(), 1, rtol=0, atol=MIX_NUMERICAL_ATOL)
    return mediane_ponderee_intervalle(frame[price_col], weights)


def analyse_mix_descriptif(lease_frame, year_pairs=MIX_YEARS,
                          strata=MIX_STRATA, price_cols=MIX_PRICE_COLS):
    """Même statistique, support commun, identité descriptive exacte.

    Renvoie trois tables agrégées : décomposition, quatre médianes
    contrefactuelles, et panel des unités communes. Aucune causalité postulée.
    """
    data = lease_frame.copy()
    required = [*MIX_UNIT_KEY, *strata, "sLeaseFrom", *price_cols]
    if data[required].isna().any().any():
        raise ValueError("Donnée manquante pour la comparaison de composition")
    data["sLeaseFrom"] = pd.to_datetime(data.sLeaseFrom, errors="raise")
    data["year"] = data.sLeaseFrom.dt.year
    if data.duplicated([*MIX_UNIT_KEY, "sLeaseFrom"]).any():
        raise ValueError("Début de bail dupliqué pour une unité")
    if not data[list(price_cols)].gt(0).all().all():
        raise ValueError("Prix non positif")

    decomposition_rows, cf_rows, panel_rows = [], [], []
    for year0, year1 in year_pairs:
        years = {year: data[data.year.eq(year)].copy() for year in (year0, year1)}
        if any(frame.empty for frame in years.values()):
            raise ValueError("Année vide")
        groups = {year: frame.groupby(list(strata), observed=True).size()
                  for year, frame in years.items()}
        common = groups[year0].index.intersection(groups[year1].index)
        if common.empty:
            raise ValueError("Aucun support commun")
        support = {year: frame[frame.set_index(list(strata)).index.isin(common)].copy()
                   for year, frame in years.items()}
        masses = {year: frame.groupby(list(strata), observed=True).size() / len(frame)
                  for year, frame in support.items()}

        # Chaque année garde ici tous ses baux, comme la médiane brute officielle.
        for col in price_cols:
            all0, all1 = [float(years[y][col].median()) for y in (year0, year1)]
            m00 = mediane_contrefactuelle(support[year0], masses[year0], col, strata)
            m01 = mediane_contrefactuelle(support[year0], masses[year1], col, strata)
            m10 = mediane_contrefactuelle(support[year1], masses[year0], col, strata)
            m11 = mediane_contrefactuelle(support[year1], masses[year1], col, strata)
            assert np.isclose(m00, support[year0][col].median(), rtol=0, atol=MIX_NUMERICAL_ATOL)
            assert np.isclose(m11, support[year1][col].median(), rtol=0, atol=MIX_NUMERICAL_ATOL)
            within = ((m10 - m00) + (m11 - m01)) / 2
            mix = ((m01 - m00) + (m11 - m10)) / 2
            extension = (all1 - m11) - (all0 - m00)
            total = all1 - all0
            assert np.isclose(total, within + mix + extension, rtol=0, atol=MIX_NUMERICAL_ATOL)
            decomposition_rows.append({
                "annee0": year0, "annee1": year1, "prix": col,
                "strates": " + ".join(strata), "cellules_communes": len(common),
                "baux_tous_0": len(years[year0]), "baux_tous_1": len(years[year1]),
                "baux_communs_0": len(support[year0]), "baux_communs_1": len(support[year1]),
                "unites_support_0": support[year0].hUnit.nunique(),
                "unites_support_1": support[year1].hUnit.nunique(),
                "baux_hors_support_0": len(years[year0]) - len(support[year0]),
                "baux_hors_support_1": len(years[year1]) - len(support[year1]),
                "mediane_tous_0": all0, "mediane_tous_1": all1,
                "variation_dollars": total, "variation_pct": 100 * total / all0,
                "distribution_intrastrate_dollars": within,
                "composition_support_commun_dollars": mix,
                "extension_support_dollars": extension,
                "distribution_intrastrate_pp": 100 * within / all0,
                "composition_support_commun_pp": 100 * mix / all0,
                "extension_support_pp": 100 * extension / all0,
                "ecart_identite_dollars": total - within - mix - extension,
            })
            cf_rows.append({"annee0": year0, "annee1": year1, "prix": col,
                            "M00_distribution0_poids0": m00,
                            "M01_distribution0_poids1": m01,
                            "M10_distribution1_poids0": m10,
                            "M11_distribution1_poids1": m11})

        # Autre question : même unité, un dernier bail observé par unité et année.
        snapshot = data[data.year.isin([year0, year1])].sort_values(
            [*MIX_UNIT_KEY, "sLeaseFrom"]
        ).drop_duplicates([*MIX_UNIT_KEY, "year"], keep="last")
        panel = snapshot[snapshot.year.eq(year0)].merge(
            snapshot[snapshot.year.eq(year1)], on=list(MIX_UNIT_KEY),
            validate="one_to_one", suffixes=("_0", "_1")
        )
        for col in price_cols:
            values0, values1 = panel[col + "_0"], panel[col + "_1"]
            panel_rows.append({"annee0": year0, "annee1": year1, "prix": col,
                               "unites_communes": len(panel),
                               "mediane_niveaux_0": values0.median(),
                               "mediane_niveaux_1": values1.median(),
                               "variation_niveaux_pct": 100 * (values1.median() / values0.median() - 1),
                               "mediane_variations_individuelles_non_annualisee_pct":
                                   100 * (values1 / values0 - 1).median()})
    return (pd.DataFrame(decomposition_rows), pd.DataFrame(cf_rows),
            pd.DataFrame(panel_rows))


mix_decomposition,mix_contrefactuels,mix_panel=analyse_mix_descriptif(leases)
labels_mix={'annee0':'Départ','annee1':'Fin','prix':'Prix','variation_dollars':'Total $',
 'distribution_intrastrate_dollars':'Distribution $','composition_support_commun_dollars':'Poids $',
 'extension_support_dollars':'Extension $','baux_tous_0':'Baux début','baux_tous_1':'Baux fin',
 'baux_communs_0':'Communs début','baux_communs_1':'Communs fin',
 'baux_hors_support_0':'Hors début','baux_hors_support_1':'Hors fin',
 'unites_support_0':'Unités début','unites_support_1':'Unités fin'}
def tableau_mix(frame,columns):
    view=frame[columns].copy()
    if 'prix' in view:view['prix']=view.prix.map({'sRent':'Contractuel','sRentEffective':'Effectif'})
    return view.rename(columns=labels_mix).round(3)
display(tableau_mix(mix_decomposition,['annee0','annee1','prix','variation_dollars',
    'distribution_intrastrate_dollars','composition_support_commun_dollars','extension_support_dollars']))
coverage=mix_decomposition.drop_duplicates(['annee0','annee1'])
display(tableau_mix(coverage,['annee0','annee1','baux_tous_0','baux_tous_1','baux_communs_0','baux_communs_1']))
display(tableau_mix(coverage,['annee0','annee1','baux_hors_support_0','baux_hors_support_1','unites_support_0','unites_support_1']))
display(mix_contrefactuels.rename(columns={**labels_mix,
 'M00_distribution0_poids0':'M00','M01_distribution0_poids1':'M01',
 'M10_distribution1_poids0':'M10','M11_distribution1_poids1':'M11'}).round(3))
display(mix_panel.rename(columns={**labels_mix,'unites_communes':'Unités communes',
 'mediane_niveaux_0':'Médiane début $','mediane_niveaux_1':'Médiane fin $',
 'variation_niveaux_pct':'Variation niveaux %',
 'mediane_variations_individuelles_non_annualisee_pct':'Médiane variations %'}).round(3))
mix_immeuble_seul,_,_=analyse_mix_descriptif(leases,strata=('sBuilding',))
display(tableau_mix(mix_immeuble_seul,['annee0','annee1','prix','distribution_intrastrate_dollars',
                         'composition_support_commun_dollars','extension_support_dollars']))
print('Identité descriptive exacte; la décomposition dépend des strates et ne démontre aucune causalité. Cible principale inchangée.')

## D — Comparer chaque unité à son bail précédent

La clé interdite n'est jamais utilisée dans le calcul. Les prédécesseurs sont définis chronologiquement, avant de filtrer les écarts. Les écarts courts sont exclus pour comparabilité selon le starter; leur exclusion ne prouve pas qu'ils sont des doublons. Les deux prix utilisent exactement les mêmes paires.

In [ ]:
def same_unit_pairs(frame, config=CONFIG):
    data = prepare_leases(frame).sort_values(UNIT_KEY + ['sLeaseFrom']).copy()
    for col in ['sRent','sRentEffective','sLeaseFrom','sTermSeq']:
        data['prev_' + col] = data.groupby(UNIT_KEY)[col].shift()
    data['gap_years'] = (data.sLeaseFrom-data.prev_sLeaseFrom).dt.days / config.days_per_year
    valid = data.gap_years.between(config.min_gap_years,config.max_gap_years,inclusive='neither')
    for col in ['sRent','sRentEffective']:
        valid &= data[col].gt(0) & data['prev_'+col].gt(0)
    exclusions = {'premiers_baux': int(data.prev_sLeaseFrom.isna().sum()),
                  'ecarts_courts': int(data.gap_years.le(config.min_gap_years).sum()),
                  'ecarts_longs': int(data.gap_years.ge(config.max_gap_years).sum()),
                  'paires_valides': int(valid.sum())}
    pairs = data.loc[valid].copy()
    pairs['contract_growth'] = ((pairs.sRent / pairs.prev_sRent)**(1/pairs.gap_years)-1)*100
    pairs['effective_growth'] = ((pairs.sRentEffective / pairs.prev_sRentEffective)**(1/pairs.gap_years)-1)*100
    if config.last_pair_per_unit:
        pairs = pairs.drop_duplicates(UNIT_KEY + ['year'],keep='last')
    return pairs, exclusions

pairs, exclusions = same_unit_pairs(leases)
assert (pairs.sTermSeq-pairs.prev_sTermSeq).eq(1).all()
display(pd.Series(exclusions, name='nombre').to_frame())
observed_annual = pairs.groupby('year').agg(paires=('hUnit','size'), unites=('hUnit','nunique'),
                 contractuel=('contract_growth','median'), effectif=('effective_growth','median'))
display(observed_annual.round(3))
fig, ax = plt.subplots()
annual.loc[2021:2025,'hausse_brute_pct'].plot(ax=ax,marker='o',linestyle='--',color=COLORS['naive'],label='Variation de la médiane brute')
observed_annual.loc[2021:2025,'contractuel'].plot(ax=ax,marker='s',color=COLORS['contractual'],label='Unité constante — contractuel')
observed_annual.loc[2021:2025,'effectif'].plot(ax=ax,marker='^',color=COLORS['effective'],label='Unité constante — effectif')
ax.axhline(0,color='#444444',linewidth=.8); ax.set(xlabel='Année du nouveau début de bail',ylabel='Variation (%)',title='Composition et concessions : mesures distinctes')
ax.set_xticks(range(2021,2026))
ax.legend(fontsize=9);fig.tight_layout();plt.show()
coverage = pairs[pairs.year.isin(TEST_YEARS)].groupby(['year','sBuilding']).size().unstack(fill_value=0)
display(coverage)

## E — Concessions : diagnostic sans double déduction

`sRentEffective` reste la valeur canonique pour la cible. Le registre de concessions aide à comprendre les crédits. Une jointure seulement sur l'unité peut multiplier les lignes. L'attribution testée exige un début de crédit dans la période du bail et une seule correspondance; les crédits non attribuables ou traversant une frontière restent visibles.

Le calcul exploratoire `sAmount * sMonths` n'est pas certifié pour tous les codes. Les montants négatifs et crédits ponctuels ne doivent pas être assimilés automatiquement à des loyers gratuits. Aucun montant n'est retranché une deuxième fois de l'effectif.

### Définition métier et limites du rapprochement

Le starter officiel, cellules 3 et 17, inclut explicitement stationnement et casier gratuits dans le loyer effectif. « Accessoire » ne signifie donc pas automatiquement « exclu de l'effectif ». Un crédit ponctuel ne doit pas être répété sur chaque mois du bail. La convention montant × mois reste un diagnostic; sa concordance avec le loyer fourni ne la certifie pas.

Le code ci-dessous ne réattribue pas automatiquement les crédits et ne modifie aucun loyer. Les tableaux produits à l'exécution dépendent exclusivement des fichiers locaux fournis.

In [ ]:
assert (leases.sConcession.eq(1) == leases.sRentEffective.lt(leases.sRent)).all()
charge_summary = concessions.groupby('sChargeCode').agg(lignes=('hUnit','size'),
                    mois_medians=('sMonths','median'), montant_median=('sAmount','median'))
display(charge_summary.round(2))
credits = concessions.copy(); credits['_credit_row'] = np.arange(len(credits))
bail_windows = leases[['hUnit','sLeaseFrom','sLeaseTo','sTermMonths','sRent','sRentEffective']].copy()
bail_windows['_lease_row'] = np.arange(len(bail_windows))
candidates = credits.merge(bail_windows,on='hUnit',how='left')
inside = candidates[(candidates.sDateFrom >= candidates.sLeaseFrom)&(candidates.sDateFrom <= candidates.sLeaseTo)].copy()
match_counts = inside.groupby('_credit_row').size()
unique = inside[inside._credit_row.map(match_counts).eq(1)].copy()
unique['credit_total_exploratoire'] = unique.sAmount*unique.sMonths
credit_totals = unique.groupby('_lease_row').credit_total_exploratoire.sum()
implied_totals = (leases.sRentEffective-leases.sRent)*leases.sTermMonths
reconstructed = pd.Series(np.arange(len(leases))).map(credit_totals).fillna(0)
reconciliation_error = (reconstructed-implied_totals.reset_index(drop=True)).abs()
concession_checks = {
 'concessions_source':len(credits), 'jointure_unite_seule_lignes':len(candidates),
 'concessions_match_unique':len(unique), 'concessions_sans_match':len(credits)-len(match_counts),
 'concessions_match_ambigu':int(match_counts.gt(1).sum()),
 'repetitions_unite_code_debut':int(credits.duplicated(['hUnit','sChargeCode','sDateFrom']).sum()),
 'montants_negatifs':int(credits.sAmount.lt(0).sum()),
 'baux_reconcilies_a_un_dollar_incluant_sans_credit':int(reconciliation_error.le(1).sum()),
 'erreur_absolue_max_totale_dollars':float(reconciliation_error.max()),
 'debuts_credit_apres_2025':int(credits.sDateFrom.gt(pd.Timestamp('2025-12-31')).sum())}
display(pd.Series(concession_checks,name='valeur').round(2).to_frame())
unmatched = credits[~credits._credit_row.isin(match_counts.index)].copy()
unit_bounds = leases.groupby('hUnit').agg(first=('sLeaseFrom','min'),last=('sLeaseTo','max'))
unmatched = unmatched.join(unit_bounds,on='hUnit')
unmatched['position'] = np.select([unmatched['first'].isna(),unmatched.sDateFrom.lt(unmatched['first']),
                                  unmatched.sDateFrom.gt(unmatched['last'])],
                                 ['unite_absente','avant_premier_bail','apres_dernier_bail'],default='entre_baux')
credit_positions = unmatched.groupby('position').agg(credits=('hUnit','size'),unites=('hUnit','nunique'))
credit_positions = credit_positions[credit_positions.unites.ge(CONFIG.min_output_pairs)]
# Only publish sufficiently large groups; suppressed categories have no values or labels.
credit_audit = {'unmatched_count':len(unmatched),'unmatched_units':int(unmatched.hUnit.nunique()),
               'unmatched_start_after_2025':int(unmatched.sDateFrom.gt(pd.Timestamp('2025-12-31')).sum()),
               'matched_within_lease_full_interval':int(unique.sDateTo.le(unique.sLeaseTo).sum()),
               'unmatched_exploratory_amount_months':float((unmatched.sAmount*unmatched.sMonths).sum()),
               'leases_with_credit_reconciled_1_dollar':int((reconciliation_error.le(1)&leases.sConcession.eq(1).reset_index(drop=True)).sum()),
               'leases_without_credit_reconciled_1_dollar':int((reconciliation_error.le(1)&leases.sConcession.eq(0).reset_index(drop=True)).sum()),
               'leases_without_credit_not_reconciled_1_dollar':int((reconciliation_error.gt(1)&leases.sConcession.eq(0).reset_index(drop=True)).sum()),
               'leases_with_credit_not_reconciled_1_dollar':int((reconciliation_error.gt(1)&leases.sConcession.eq(1).reset_index(drop=True)).sum()),
               'positions_published':credit_positions.reset_index().to_dict('records'),
               'positions_suppressed':int(unmatched.position.nunique()-len(credit_positions))}
assert credit_audit['leases_with_credit_reconciled_1_dollar']+credit_audit['leases_without_credit_reconciled_1_dollar']==concession_checks['baux_reconcilies_a_un_dollar_incluant_sans_credit']
display(credit_positions);display(pd.Series({k:v for k,v in credit_audit.items() if k not in ['positions_published']},name='valeur').to_frame())
# As-of 2025 exposure excludes credits whose application starts in 2026.
exposed_units = unmatched.loc[unmatched.sDateFrom.le(pd.Timestamp('2025-12-31')),'hUnit'].unique()
reconciliation_units = leases.loc[reconciliation_error.to_numpy()>1,'hUnit'].unique()
credit_sensitivity_rows=[]
for label,units in [('toutes_paires',[]),('sans_unites_credit_non_attribue_avant_2026',exposed_units),
                    ('sans_unites_reconciliation_incomplete',reconciliation_units)]:
    population = pairs[pairs.year.eq(2025)&~pairs.hUnit.isin(units)]
    credit_sensitivity_rows.append({'scenario':label,'paires':len(population),'unites':int(population.hUnit.nunique()),
                                    'mediane_effective_pct':float(population.effective_growth.median())})
credit_sensitivity=pd.DataFrame(credit_sensitivity_rows)
assert credit_sensitivity.unites.ge(CONFIG.min_output_pairs).all()
display(credit_sensitivity.round(3))

concession_annual = leases.groupby('year').agg(part_concession=('sConcession','mean'),
                      contractuel=('sRent','median'), effectif=('sRentEffective','median'))
fig, ax = plt.subplots()
(concession_annual.part_concession*100).plot(ax=ax,marker='o',color=COLORS['effective'])
ax.set(xlabel='Année de début',ylabel='Baux avec concession (%)',ylim=(0,100),title='Fréquence des concessions dans les baux observés')
fig.tight_layout();plt.show()
# Diagnostic descriptif demandé après relecture indépendante, sans réattribution.
credit_between=unmatched[unmatched.position.eq('entre_baux')]
near=credit_between[['hUnit','_credit_row','sDateFrom']].merge(
    bail_windows[['hUnit','sLeaseFrom','sLeaseTo']],on='hUnit',how='left')
near['distance_jours']=np.minimum((near.sDateFrom-near.sLeaseFrom).dt.days.abs(),
                                  (near.sDateFrom-near.sLeaseTo).dt.days.abs())
nearest=near.groupby('_credit_row').distance_jours.min()
assert credit_between.hUnit.nunique()>=CONFIG.min_output_pairs
credit_between_summary={'credits':len(credit_between),'unites':int(credit_between.hUnit.nunique()),
    'distance_frontiere_mediane_jours':float(nearest.median()),
    'distance_frontiere_q1_jours':float(nearest.quantile(.25)),
    'distance_frontiere_q3_jours':float(nearest.quantile(.75))}
display(pd.Series(credit_between_summary,name='valeur').to_frame())
credit_hypotheses=[]
credit_once=unique.groupby('_lease_row').sAmount.sum()
rent_only=unique[unique.sChargeCode.isin(['PromoPay','freerent'])].groupby('_lease_row').credit_total_exploratoire.sum()
with_credit=leases.sConcession.eq(1).reset_index(drop=True)
for label,totals in [('montant_une_fois',credit_once),('tous_codes_montant_fois_mois',credit_totals),
                     ('PromoPay_freerent_montant_fois_mois',rent_only)]:
    reconstructed_variant=pd.Series(np.arange(len(leases))).map(totals).fillna(0)
    errors=(reconstructed_variant-implied_totals.reset_index(drop=True)).abs()[with_credit]
    credit_hypotheses.append({'hypothese':label,'baux_avec_concession':int(with_credit.sum()),
        'unites':int(leases.loc[with_credit.to_numpy(),'hUnit'].nunique()),
        'baux_ecart_au_plus_1_dollar':int(errors.le(1).sum()),
        'ecart_total_absolu_median_dollars':float(errors.median())})
credit_hypotheses=pd.DataFrame(credit_hypotheses)
assert credit_hypotheses.unites.ge(CONFIG.min_output_pairs).all()
display(credit_hypotheses.round(2))
print('Hypothèses de rapprochement, aucune formule comptable certifiée; effectif fourni inchangé.')

## F — Renouvellements, relocations et marchés

Comparer les deux cas sur les paires valides. Afficher des agrégats seulement au-dessus du seuil de couverture configuré; une cellule trop petite est omise, pas remplacée par zéro. L'applicabilité des règles juridiques de chaque immeuble n'est pas déduite de son premier bail observé.

### Ottawa : branches réglementaires, sans conformité individuelle présumée

| Situation à établir | Lecture réglementaire | Preuve absente des données fournies |
|---|---|---|
| Locataire en place, location assujettie à la ligne directrice | Ligne directrice Ontario applicable sous conditions; préavis et délais applicables, décisions particulières distinctes | Régime, loyer légal de référence, avis et décisions éventuelles |
| Exemption de ligne directrice | La première occupation résidentielle du bâtiment ou de l’addition est déterminante selon les conditions; le premier bail observé ne suffit pas | Justificatif de première occupation; autres obligations toujours pertinentes |
| Véritable nouveau locataire | Loyer d’entrée convenu pour la nouvelle location; une relocation enregistrée ne qualifie pas juridiquement une cession ou continuation | Nature du changement de locataire |
| Rabais et services | Le loyer économique effectif n'est pas nécessairement le loyer légal; la forme écrite des rabais et les services inclus comptent | Convention, clauses et périmètre des avantages |
| Éventuel avis de réduction lié aux taxes | Vérifier l'avis municipal applicable | Avis propre à l'immeuble, non obtenu |

Les ventilations provinciales sont descriptives; le score principal est global. Les références officielles sont liées ci-dessous. Cette analyse ne détermine pas le loyer légal d'un bail individuel.

In [ ]:
slice_2025 = pairs[pairs.year.eq(2025)].groupby(['sState','sRenewal']).agg(
    paires=('hUnit','size'),contractuel=('contract_growth','median'),effectif=('effective_growth','median'))
slice_2025 = slice_2025[slice_2025.paires.ge(CONFIG.min_output_pairs)]
display(slice_2025.round(3))
overall_type_2025 = pairs[pairs.year.eq(2025)].groupby('sRenewal').agg(
    paires=('hUnit','size'),unites=('hUnit','nunique'),
    contractuel=('contract_growth','median'),effectif=('effective_growth','median'))
overall_type_2025.index = overall_type_2025.index.map({0:'Relocations',1:'Renouvellements'})
display(overall_type_2025.round(3))
fig, ax=plt.subplots()
slice_2025[['contractuel','effectif']].plot.bar(ax=ax,color=[COLORS['contractual'],COLORS['effective']],rot=0)
ax.set_xticklabels([state+'\n'+('Renouvellement' if renewal else 'Relocation') for state,renewal in slice_2025.index])
ax.set(title='Paires 2025 : province et type de bail',xlabel='Province et type de bail',ylabel='Médiane annualisée (%)')
fig.tight_layout();plt.show()

## G — Source externe réellement intégrée : IPC loyer

Source : Statistique Canada **18-10-0004-01**, composante `Rent`, Québec et Ontario, indice non désaisonnalisé 2002=100. Deux géographies provinciales sont utilisées comme repères; elles ne représentent pas les concessions ni le marché d'un immeuble.

Le ZIP public est archivé localement, et le sous-ensemble public utilisé est inclus dans la remise. Les données de novembre 2022, 2023, 2024 et 2025 ont été publiées respectivement les **21 décembre 2022, 19 décembre 2023, 17 décembre 2024 et 15 décembre 2025**. Les valeurs de novembre de l'année précédente servent au dénominateur; leur ratio est calculé dans cette cellule. Aucun décembre publié en janvier n'est utilisé.

La table actuelle a été téléchargée le 3 octobre 2026 : les dates de première diffusion sont vérifiées, mais les valeurs ne proviennent pas de quatre archives historiques séparées. Le test IPC comporte donc une limite de millésime/révision; il n'est pas une simulation parfaite en temps réel. La prévision interne n'a pas cette dépendance externe.

SCHL 2025 (publication le 11 décembre 2025) : extrait public de la page officielle inclus et affiché ci-dessous. Les marchés Montréal RMR et Ottawa (portion Ontario) et les hausses moyennes de deux chambres diffèrent de notre médiane effective, toutes chambres, six immeubles. Cet extrait sert à confronter le contexte, sans fusion des cibles ni gain prédictif allégué. L’édition 2025 du classeur officiel a maintenant été consultée et ses indicateurs utiles sont présentés dans la section P. Les données SCHL demeurent du contexte et aucune série historique n’est ajoutée au moteur. Une enquête 2023 publiée le 31 janvier 2024 ne peut pas prédire 2024 à fin 2023; publication 2024 le 17 décembre 2024 utilisable à fin 2024. TAL/Ontario : contexte et exceptions discutés en références, pas coefficients automatiques de hausse. Les avis d’attribution prescrits pour les données externes et les licences correspondantes sont enregistrés dans `externes/registre-sources.json`. StatCan 46-10-0092-01 et les achèvements : non intégrés dans cette version.

In [ ]:
CPI_RELEASES = {
 2018: ('2018-12-19','https://www150.statcan.gc.ca/n1/daily-quotidien/181219/dq181219a-eng.pdf'),
 2019: ('2019-12-18','https://www150.statcan.gc.ca/n1/daily-quotidien/191218/dq191218a-eng.htm'),
 2020: ('2020-12-16','https://www150.statcan.gc.ca/n1/daily-quotidien/201216/dq201216a-eng.htm'),
 2021: ('2021-12-15','https://www150.statcan.gc.ca/n1/daily-quotidien/211215/dq211215a-eng.htm'),
 2022: ('2022-12-21','https://www150.statcan.gc.ca/n1/daily-quotidien/221221/dq221221a-eng.htm'),
 2023: ('2023-12-19','https://www150.statcan.gc.ca/n1/daily-quotidien/231219/dq231219a-eng.htm'),
 2024: ('2024-12-17','https://www150.statcan.gc.ca/n1/daily-quotidien/241217/dq241217a-eng.htm'),
 2025: ('2025-12-15','https://www150.statcan.gc.ca/n1/daily-quotidien/251215/dq251215a-eng.htm')}
public_slice_path = PUBLIC_DIR / 'statcan-rent-november-source.csv'
metadata_path = PUBLIC_DIR / 'statcan-source-metadata.json'
if public_slice_path.exists() and pd.read_csv(public_slice_path).REF_DATE.min() <= '2017-11':
    cpi_source = pd.read_csv(public_slice_path)
    source_meta = json.loads(metadata_path.read_text(encoding='utf-8'))
else:
    if not PUBLIC_ARCHIVE.exists():
        raise FileNotFoundError('Sous-ensemble public IPC manquant; fournir les fichiers publics externes livrés avec le notebook.')
    selected = []
    with zipfile.ZipFile(PUBLIC_ARCHIVE) as public_archive:
        with public_archive.open('18100004.csv') as stream:
            for chunk in pd.read_csv(stream,chunksize=200000,low_memory=False):
                mask = chunk.GEO.isin(['Quebec','Ontario']) & chunk['Products and product groups'].eq('Rent')
                mask &= chunk.REF_DATE.str.endswith('-11') & chunk.REF_DATE.between('2017-11','2025-11')
                selected.append(chunk.loc[mask])
    cpi_source = pd.concat(selected,ignore_index=True)
    cpi_source.to_csv(public_slice_path,index=False)
    source_meta = json.loads(PUBLIC_METADATA.read_text(encoding='utf-8'))
    source_meta['public_slice_sha256'] = hashlib.sha256(public_slice_path.read_bytes()).hexdigest()
    source_meta['selection'] = 'Rent; Quebec/Ontario; novembre 2017-2025; valeurs inchangées'
    metadata_path.write_text(json.dumps(source_meta,ensure_ascii=False,indent=2),encoding='utf-8')
assert hashlib.sha256(public_slice_path.read_bytes()).hexdigest()==source_meta['public_slice_sha256']
assert len(cpi_source)==18 and not cpi_source.duplicated(['GEO','REF_DATE']).any()
assert cpi_source.VALUE.notna().all() and cpi_source.VALUE.gt(0).all()
assert cpi_source.STATUS.fillna('').eq('').all(), 'Statut public à examiner'
signals = []
for state in ['Quebec','Ontario']:
    source_values = cpi_source[cpi_source.GEO.eq(state)].set_index('REF_DATE').VALUE
    for year,(released,url) in CPI_RELEASES.items():
        signals.append({'source':'Statistique Canada','series':'18-10-0004-01 / Rent','geography':state,
             'reference_period':f'{year}-11','release_date':released,
             'value':float((source_values[f'{year}-11']/source_values[f'{year-1}-11']-1)*100),
             'unit':'variation sur 12 mois (%)','release_url':url,'vintage':source_meta['vintage']})
external = pd.DataFrame(signals)
external['release_date'] = pd.to_datetime(external.release_date)
external['reference_date'] = pd.to_datetime(external.reference_period+'-01')
external.to_csv(PUBLIC_DIR/'ipc-signaux.csv',index=False)
display(external[['geography','reference_period','release_date','value']].assign(value=external.value.round(3)))
print('Public :',len(cpi_source),'indices,',len(external),'signaux; téléchargement',source_meta['download_date'])
cmhc_context=pd.read_csv(PUBLIC_DIR/'schl-contexte-2025.csv')
assert cmhc_context.release_date.eq('2025-12-11').all()
context_2025_summary=cmhc_context[cmhc_context.geographie.isna()].copy()
assert len(context_2025_summary)==6
display(Markdown('**Synthèse de six indicateurs SCHL 2025.** Le tableau complet 2024/2025 avec qualité par année est présenté en section P.'))
display(context_2025_summary[['geography','metric','reference_period','value','unit']])
# A/B/C remain compared on identical internal targets; CMHC is a contextual external check.

## H — Moteur commun et trois familles de prévision

Chaque méthode filtre les contrats **avant** de construire les paires. Les annonces et les sources externes sont également filtrées à la coupure dans le moteur. Les caractéristiques d'unités futures et les étiquettes de l'année à prévoir ne participent pas à la prévision.

**A :** réutiliser la dernière distribution annuelle effective comparable. **B :** recentrer la distribution de référence par province/type de bail sur la médiane de ses médianes annuelles des trois dernières années; si moins de 30 paires ou deux années dans le segment, repli au type de bail puis au global. **C :** même B avec un ancrage IPC à poids fixe 0,5, non réglé sur les résultats. Un seul prédicteur externe est testé.

Les distributions prédites sont réunies avant de prendre leur médiane. Le mélange central conserve les parts de transitions de la dernière année disponible. Les modèles ne prédisent pas quels locataires changeront de bail ni l'occupation future. Les annonces sont disponibles pour diagnostic mais ne sont pas un prédicteur retenu; aucun gain ne leur est attribué.

In [ ]:
def history_at(frame, cutoff, config):
    data=prepare_leases(frame)
    known=data[data.sSignDate.le(cutoff)&data.sLeaseFrom.le(cutoff)].copy()
    if config.availability=='conservative':
        known=known[known.sLeaseTo.le(cutoff)].copy()
    return known

def historical_location(history, state, renewal, config, price_col):
    recent=history[history.year.ge(int(history.year.max())-config.history_years+1)]
    scopes=[recent[recent.sState.eq(state)&recent.sRenewal.eq(renewal)],
            recent[recent.sRenewal.eq(renewal)], recent]
    labels=['province_type','type','global']
    for scope,label in zip(scopes,labels):
        annual_medians=scope.groupby('year')[price_col].median()
        if (len(scope)>=config.min_segment_pairs and len(annual_medians)>=2) or label=='global':
            if annual_medians.empty: raise ValueError('Historique comparable insuffisant')
            return float(annual_medians.median()),label
    raise ValueError('Historique comparable insuffisant')

def forecast_engine(frame, asking_frame, external_frame, year, method='A_derniere',config=CONFIG,
                    price_col='effective_growth'):
    cutoff=pd.Timestamp(year=year-1,month=12,day=31)
    known=history_at(frame,cutoff,config)
    hist,_=same_unit_pairs(known,config)
    if hist.empty:raise ValueError('Aucune paire historique éligible')
    last_year=int(hist.year.max())
    reference=hist[hist.year.eq(last_year)].copy()
    available_asking=asking_frame[asking_frame.month_end.le(cutoff)].copy()
    available_external=(external_frame[external_frame.release_date.le(cutoff)&external_frame.reference_date.le(cutoff)].copy()
                        if external_frame is not None else pd.DataFrame())
    predicted=reference[price_col].to_numpy().copy()
    if config.clip_quantiles!=(0.0,1.0):
        lo,hi=hist[price_col].quantile(config.clip_quantiles)
        predicted=np.clip(predicted,lo,hi)
    if method!='A_derniere':
        for (state,renewal),group in reference.groupby(['sState','sRenewal']):
            location,scope=historical_location(hist,state,renewal,config,price_col)
            indexes=reference.index.get_indexer(group.index)
            if method=='C_B_plus_IPC':
                if available_external.empty:raise ValueError('IPC publié manquant à la coupure')
                province=available_external[available_external.geography.eq(state)].sort_values('release_date')
                if province.empty:raise ValueError('IPC provincial manquant à la coupure')
                cpi=float(province.iloc[-1].value)
                location=(1-config.cpi_anchor_weight)*location+config.cpi_anchor_weight*cpi
            predicted[indexes]+=location-float(group[price_col].median())
    if method not in METHODS:raise ValueError('Méthode inconnue')
    assert known.sLeaseFrom.max()<=cutoff and known.sSignDate.max()<=cutoff
    assert available_asking.empty or available_asking.month_end.max()<=cutoff
    assert available_external.empty or available_external.release_date.max()<=cutoff
    reference['predicted_growth']=predicted
    return {'year':year,'cutoff':str(cutoff.date()),'method':method,'estimate_pct':float(np.median(predicted)),
            'history_pairs':len(hist),'reference_pairs':len(reference),'reference_units':int(reference.hUnit.nunique()),
            'reference_year':last_year,'reference_buildings':int(reference.sBuilding.nunique()),
            'eligible_asking_rows':len(available_asking),'external_rows_available':len(available_external),
            'availability':config.availability,'samples':reference}

def observed_target(frame,year,config=CONFIG,price_col='effective_growth'):
    # Calcul séparé, après la prévision. Aucun passage de cette table au moteur.
    data=prepare_leases(frame)
    observed,_=same_unit_pairs(data[data.sLeaseFrom.le(pd.Timestamp(year,12,31))],config)
    target=observed[observed.year.eq(year)]
    if target.empty:raise ValueError('Aucune observation cible')
    return {'observed_pct':float(target[price_col].median()),'observed_pairs':len(target),
            'observed_units':int(target.hUnit.nunique()),'observed_buildings':int(target.sBuilding.nunique())}

def evaluate_method(frame,year,method,config=CONFIG):
    forecast=forecast_engine(frame,asking,external,year,method,config)
    actual=observed_target(frame,year,config)
    result={k:v for k,v in forecast.items() if k!='samples'}
    result.update(actual)
    result['error_pp']=result['estimate_pct']-result['observed_pct']
    result['absolute_error_pp']=abs(result['error_pp'])
    return result

def estimate_2026(leases, asking, external=None):
    '''Retourne la médiane prévue de hausse effective annualisée 2026, en %.'''
    return forecast_engine(leases,asking,external,2026,SELECTED_METHOD,CONFIG)['estimate_pct']

def backtest(leases, target_year):
    '''Interface officielle : méthode comme à fin target_year-1, puis résultat observé séparé.'''
    return evaluate_method(leases,int(target_year),SELECTED_METHOD,CONFIG)

print('Interfaces officielles :',inspect.signature(estimate_2026),inspect.signature(backtest))

## I — Comparaison exploratoire 2023–2025

Même cible, mêmes coupures et mêmes dénominateurs pour les trois méthodes. MAE = moyenne des erreurs absolues en points de pourcentage, avec poids égal par année. Pour retenir une variante plus complexe, exiger au moins 0,25 point de gain moyen et une amélioration sur deux années sur trois; sinon garder A. Ce seuil pratique est un choix annoncé, pas un seuil officiel du jury.

Les résultats ont été consultés pendant le développement; aucune année n'est présentée comme un test indépendant jamais vu. Le contrôle IPC utilise un millésime actuel avec premières dates de diffusion vérifiées, ce qui limite son interprétation historique.

In [ ]:
experiment_rows=[evaluate_method(leases,year,method) for method in METHODS for year in TEST_YEARS]
experiments=pd.DataFrame(experiment_rows)
summary=experiments.groupby('method').absolute_error_pp.agg(MAE_pp='mean',erreur_max_pp='max')
baseline_errors=experiments[experiments.method.eq('A_derniere')].set_index('year').absolute_error_pp
for method in METHODS:
    errs=experiments[experiments.method.eq(method)].set_index('year').absolute_error_pp
    summary.loc[method,'annees_ameliorees']=int(errs.lt(baseline_errors).sum())
    summary.loc[method,'gain_MAE_pp']=float(baseline_errors.mean()-errs.mean())
eligible=summary[(summary.gain_MAE_pp>=CONFIG.complexity_gain_pp)&(summary.annees_ameliorees>=2)]
SELECTED_METHOD=eligible.MAE_pp.idxmin() if not eligible.empty else 'A_derniere'
display(experiments[['method','year','estimate_pct','observed_pct','error_pp','absolute_error_pp',
                     'reference_pairs','observed_pairs','observed_buildings']].round(3))
display(summary.round(3)); print('Méthode retenue :',SELECTED_METHOD)
fig,ax=plt.subplots()
comparison=experiments.pivot(index='year',columns='method',values='estimate_pct')
comparison['Observe']=experiments[experiments.method.eq('A_derniere')].set_index('year').observed_pct
comparison.plot.bar(ax=ax,color=['#245780','#a16b18','#8b728d','#444444'],rot=0)
ax.set(xlabel='Année prédite',ylabel='Médiane effective annualisée (%)',title='Prévisions et observations — coupures à fin d’année précédente')
ax.legend(fontsize=8,loc='upper left',bbox_to_anchor=(1.01,1));fig.tight_layout();plt.show()
final_backtests=pd.DataFrame([backtest(leases,y) for y in TEST_YEARS])

# L'objectif de décision est explicite : MAE + seuil pratique de complexité.
# La pire erreur est descriptive, pas un nouveau critère de sélection a posteriori.
risk_tradeoff = {
    'selection_objective':'MAE exploratoire; gain minimal de complexité et deux années améliorées',
    'gain_MAE_B_vs_A_pp':float(summary.loc['B_robuste_segmentee','gain_MAE_pp']),
    'reduction_erreur_max_B_vs_A_pp':float(summary.loc['A_derniere','erreur_max_pp']-summary.loc['B_robuste_segmentee','erreur_max_pp']),
    'reduction_erreur_max_B_vs_A_pct':float(100*(1-summary.loc['B_robuste_segmentee','erreur_max_pp']/summary.loc['A_derniere','erreur_max_pp'])),
    'independent_validation':False,
    'meaning':'B réduit la pire erreur observée; A est retenu par la règle annoncée, sans supériorité universelle.'}
display(Markdown(f"**Compromis de risque :** B réduit la pire erreur observée de {risk_tradeoff['reduction_erreur_max_B_vs_A_pp']:.3f} point ({risk_tradeoff['reduction_erreur_max_B_vs_A_pct']:.1f} %), mais son gain MAE reste {risk_tradeoff['gain_MAE_B_vs_A_pp']:.3f} point. Le classement dépend de l'objectif; ces trois années ont été consultées pendant le développement. Aucun critère minimax n'est substitué après coup à la règle annoncée."))

## I bis — Sensibilité historique 2019–2022

Même cible, mêmes filtres et mêmes dates de coupure que la comparaison principale. Les valeurs publiques IPC sont étendues aux novembre 2017–2025; les publications de novembre 2018–2021 datent des 19 décembre 2018, 18 décembre 2019, 16 décembre 2020 et 15 décembre 2021. Les trois familles sont comparées sur les mêmes années, sans modifier la règle de sélection sur 2023–2025.

2018 est exclue de cette prévision : aucune paire comparable antérieure n'est disponible. Les années supplémentaires portent surtout sur les premiers immeubles québécois, avec peu de référence au début. Elles éclairent la robustesse historique; elles ne sont pas un test indépendant et ne valident pas la généralisation à Ottawa ou aux immeubles récents. Le millésime actuel des indices conserve ses limites.

In [ ]:
older_experiments=pd.DataFrame([evaluate_method(leases,y,m) for m in METHODS for y in range(2019,2023)])
assert older_experiments.groupby('year').observed_pairs.nunique().eq(1).all()
assert older_experiments.groupby('year').reference_pairs.nunique().eq(1).all()
display(older_experiments[['method','year','estimate_pct','observed_pct','absolute_error_pp','reference_pairs','observed_pairs']].round(3))
extended_summary=pd.concat([older_experiments.assign(period='2019_2022'),experiments.assign(period='2023_2025')]).groupby(['period','method']).absolute_error_pp.agg(MAE_pp='mean',annees='size')
display(extended_summary.round(3))
print('Sélection principale inchangée :',SELECTED_METHOD)

## J — Sensibilités et tests d'accès au futur

Contrôler l'exclusion des données futures en perturbant les loyers après chaque coupure; la prévision doit rester identique. Ce test vérifie le filtrage informatique, pas la date réelle de saisie des crédits historiques. Contrôler aussi l'invariance à l'ordre des lignes.

Les sensibilités de maturité, fenêtre, bornes et unité/an restent des analyses de robustesse, pas de nouvelles familles optimisées. L'exclusion d'un immeuble et la pondération égale des bâtiments modifient la population/pondération : elles ne sont pas la cible principale.

Correction après relecture séparée : le paramètre de profondeur historique n’agit que sur B/C; les fenêtres 2/4 ans ne sont donc pas présentées comme sensibilités actives de A. Une MAE n’est affichée que si toutes les années ont au moins 30 unités de référence et de cible. Le scénario de maturité strict reste un diagnostic de couverture; son score insuffisamment couvert est masqué. La winsorisation laissant une médiane intérieure inchangée est une propriété mathématique, pas une certification de qualité des loyers.


In [ ]:
for year in TEST_YEARS:
    cutoff=pd.Timestamp(year-1,12,31)
    perturbed=leases.copy()
    future=perturbed.sLeaseFrom.gt(cutoff)|perturbed.sSignDate.gt(cutoff)
    perturbed.loc[future,['sRent','sRentEffective']]*=10
    for method in METHODS:
        original=forecast_engine(leases,asking,external,year,method)['estimate_pct']
        changed=forecast_engine(perturbed,asking,external,year,method)['estimate_pct']
        assert np.isclose(original,changed), 'Accès au futur détecté'
    altered_external=external.copy()
    altered_external.loc[altered_external.release_date.gt(cutoff),'value']=999
    assert np.isclose(forecast_engine(leases,asking,external,year,'C_B_plus_IPC')['estimate_pct'],
                      forecast_engine(leases,asking,altered_external,year,'C_B_plus_IPC')['estimate_pct'])
assert np.isclose(estimate_2026(leases,asking,external),estimate_2026(leases.sample(frac=1,random_state=17),asking,external))

sensitivity_configs={
 'principal':CONFIG,
 'baux_termines':replace(CONFIG,availability='conservative'),
 'derniere_paire_unite_an':replace(CONFIG,last_pair_per_unit=True),
 'ecarts_0.75_1.5':replace(CONFIG,min_gap_years=.75,max_gap_years=1.5),
 'extremes_winsor_1_99':replace(CONFIG,clip_quantiles=(.01,.99))}
# history_years agit seulement sur B/C, jamais sur A.
if SELECTED_METHOD != 'A_derniere':
    sensitivity_configs.update({'historique_2_ans':replace(CONFIG,history_years=2),
                                'historique_4_ans':replace(CONFIG,history_years=4)})
sensitivity_rows=[]
for label,config in sensitivity_configs.items():
    tests=[evaluate_method(leases,y,SELECTED_METHOD,config) for y in TEST_YEARS]
    final=forecast_engine(leases,asking,external,2026,SELECTED_METHOD,config)
    min_units=min(r['reference_units'] for r in tests)
    score_admissible=(min_units>=CONFIG.min_output_pairs and
                      min(r['observed_units'] for r in tests)>=CONFIG.min_output_pairs)
    sensitivity_rows.append({'scenario':label,'estimation_2026_pct':final['estimate_pct'],
            'MAE_2023_2025_pp':float(np.mean([r['absolute_error_pp'] for r in tests])) if score_admissible else np.nan,
            'paires_reference':final['reference_pairs'], 'unites_reference_2026':final['reference_units'],
            'unites_min_backtest':min_units,
            'statut_estimation':'couverture_suffisante' if final['reference_units']>=CONFIG.min_output_pairs else 'couverture_insuffisante',
            'statut_score':'exploratoire' if score_admissible else 'couverture_insuffisante'})
sensitivities=pd.DataFrame(sensitivity_rows)
display(sensitivities.round(3))

def weighted_median(values,weights):
    order=np.argsort(values); values=np.asarray(values)[order];weights=np.asarray(weights)[order]
    cumulative=np.cumsum(weights)/np.sum(weights)
    return float(values[np.searchsorted(cumulative,.5)])

final_forecast=forecast_engine(leases,asking,external,2026,SELECTED_METHOD)
forecast_samples=final_forecast['samples']
base_weights=np.ones(len(forecast_samples))
building_counts=forecast_samples.groupby('sBuilding').size()
equal_building_weights=1/forecast_samples.sBuilding.map(building_counts).to_numpy()
weighting_rows=[{'scenario':'poids_identiques_par_immeuble',
                'estimation_2026_pct':weighted_median(forecast_samples.predicted_growth,equal_building_weights),
                'unites_reference_2026':int(forecast_samples.hUnit.nunique())}]
for building in sorted(leases.sBuilding.unique()):
    subset=leases[leases.sBuilding.ne(building)]
    excluded_forecast=forecast_engine(subset,asking,external,2026,SELECTED_METHOD)
    weighting_rows.append({'scenario':'exclusion_'+building,'estimation_2026_pct':excluded_forecast['estimate_pct'],
                           'unites_reference_2026':excluded_forecast['reference_units']})
for name,shift in [('moins_relocations',-.10),('plus_relocations',.10)]:
    share=float(forecast_samples.sRenewal.eq(0).mean())
    target_share=float(np.clip(share+shift,0.05,0.95))
    weights=np.where(forecast_samples.sRenewal.eq(0),target_share/share,(1-target_share)/(1-share))
    weighting_rows.append({'scenario':name,'estimation_2026_pct':weighted_median(forecast_samples.predicted_growth,weights),
                           'unites_reference_2026':int(forecast_samples.hUnit.nunique())})
weighting_scenarios=pd.DataFrame(weighting_rows)
display(weighting_scenarios.round(3))
print('Contrôles temporels et invariance à l’ordre : réussis.')

## K — Estimation 2026, scénarios de composition et ventilations

La plage principale correspond uniquement à une variation configurée de la part des relocations, avec les taux historiques fixés. C'est une hypothèse de composition, pas un intervalle de confiance ni une garantie de couverture prédictive. De nouveaux chocs de marché peuvent se trouver hors de cette plage.

Les autres sensibilités restent séparées. Une estimation ne contribue à leur enveloppe descriptive que si la couverture minimale configurée est atteinte. Les niveaux par immeuble, chambres ou type de bail ne fixent pas le loyer légal d'un appartement.

In [ ]:
estimate=estimate_2026(leases,asking,external)
covered_sensitivities=sensitivities[sensitivities.unites_reference_2026.ge(CONFIG.min_output_pairs)]
covered_weights=weighting_scenarios[weighting_scenarios.unites_reference_2026.ge(CONFIG.min_output_pairs)]
covered_values=np.r_[covered_sensitivities.estimation_2026_pct.to_numpy(),covered_weights.estimation_2026_pct.to_numpy()]
covered_sensitivity_low=float(covered_values.min());covered_sensitivity_high=float(covered_values.max())
excluded_sensitivities=sensitivities[sensitivities.unites_reference_2026.lt(CONFIG.min_output_pairs)]
building_forecasts=forecast_samples.groupby('sBuilding').agg(paires_reference=('hUnit','size'),unites_reference=('hUnit','nunique'),
                mediane_prevue_pct=('predicted_growth','median'))
building_forecasts=building_forecasts[building_forecasts.unites_reference.ge(CONFIG.min_output_pairs)]
bedroom_forecasts=forecast_samples.groupby(['sState','sBeds']).agg(paires_reference=('hUnit','size'),unites_reference=('hUnit','nunique'),
                mediane_prevue_pct=('predicted_growth','median'))
bedroom_forecasts=bedroom_forecasts[bedroom_forecasts.unites_reference.ge(CONFIG.min_output_pairs)]
display(building_forecasts.round(2));display(bedroom_forecasts.round(2))
segment_tables={}; suppressed_groups={}
for label,keys in [('immeuble_type',['sBuilding','sRenewal']),('immeuble_chambres',['sBuilding','sBeds']),
                   ('immeuble_chambres_type',['sBuilding','sBeds','sRenewal'])]:
    table=forecast_samples.groupby(keys).agg(paires=('hUnit','size'),unites=('hUnit','nunique'),
                                            central_pct=('predicted_growth','median'))
    keep=table.unites.ge(CONFIG.min_output_pairs)
    suppressed_groups[label]=int((~keep).sum())
    segment_tables[label]=table[keep]
    display(Markdown('**'+label+' : cellules de 30 unités minimum**'));display(table[keep].round(3))
display(pd.Series(suppressed_groups,name='cellules_omises').to_frame())
# Low/high are composition scenarios, no probability or future predictive coverage.
composition_low=float(weighting_scenarios[weighting_scenarios.scenario.isin(['moins_relocations','plus_relocations'])].estimation_2026_pct.min())
composition_high=float(weighting_scenarios[weighting_scenarios.scenario.isin(['moins_relocations','plus_relocations'])].estimation_2026_pct.max())
scenario_low=composition_low;scenario_high=composition_high

fig,ax=plt.subplots()
building_forecasts.mediane_prevue_pct.sort_values().plot.barh(ax=ax,color=COLORS['effective'])
ax.set(xlabel='Médiane effective annualisée prévue (%)',ylabel='',title='Prévision conditionnelle par immeuble — composition passée')
fig.tight_layout();plt.show()
o25=observed_annual.loc[2025]
display(Markdown(f'''**Estimation centrale 2026 : {estimate:.1f} %**, méthode `{SELECTED_METHOD}`.

Scénarios de composition (part des relocations ±10 points) : **{scenario_low:.2f} à {scenario_high:.2f} %**, sans niveau de confiance statistique. Le scénario de maturité sur 7 unités est un diagnostic de couverture insuffisante; il ne définit aucune borne principale.

En 2025, **{int(o25.paires)} transitions / {int(o25.unites)} unités** donnent une médiane contractuelle de **{o25.contractuel:.2f} %** et effective de **{o25.effectif:.2f} %**. Ces taux décrivent l'historique et ne sont pas le résultat prévisionnel par définition.

MAE historique de la méthode retenue : **{summary.loc[SELECTED_METHOD,'MAE_pp']:.2f} points**, sur seulement trois années exploratoires. Les scénarios de maturité, le millésime IPC et le changement de composition limitent la portée de la conclusion.'''))

## M — Robustesse des définitions : cible, prix et population

Ces variantes sont annoncées comme diagnostics avant calcul. La définition principale reste la médiane effective annualisée, transitions comparables à poids égal. Moyenne, contractuel et annonces répondent à d'autres questions; leurs erreurs ne servent pas à choisir la définition qui donne un chiffre préféré. Les premiers baux ne deviennent pas des transitions comparables parce que leur prix est connu.

La moyenne donne davantage de poids aux fortes variations. La variation non annualisée compare les prix mais change le traitement des périodes. Retirer les transitions avec concession dans l'un des deux baux modifie la population; ce n'est pas corriger comptablement l'effectif. Les bornes temporelles plus étroites ont un effet de sélection.

Annonces : apparier propriété/unité et même mois de l'année précédente, sans compléter les mois manquants. Chaque observation mensuelle comparable contribue; le même logement peut contribuer plusieurs mois. Le panel est incomplet et sélectionné par les disponibilités d'annonces. Une croissance affichée n'est pas le prix négocié net de concessions. Un résultat n'est affiché que si au moins 30 unités distinctes contribuent. Les comparaisons de niveau affiché/effectif sur des populations différentes seraient trompeuses.


In [ ]:
# Toutes les variantes portent sur le dernier historique au 31 décembre 2025.
known_2025=history_at(leases,pd.Timestamp('2025-12-31'),CONFIG)
robust_pairs,_=same_unit_pairs(known_2025)
r25=robust_pairs[robust_pairs.year.eq(2025)].copy()
definition_rows=[]
def add_definition(label,frame,column,aggregation='median',meaning='même population'):
    enough=frame.hUnit.nunique()>=CONFIG.min_output_pairs
    definition_rows.append({'definition':label,'paires':len(frame),'unites':int(frame.hUnit.nunique()),
                            'valeur_pct':float(getattr(frame[column],aggregation)()) if enough else None,
                            'statut':'calculable' if enough else 'effectif_insuffisant','interpretation':meaning})
add_definition('effectif_mediane_annualisee',r25,'effective_growth')
add_definition('effectif_moyenne_annualisee',r25,'effective_growth','mean')
add_definition('contractuel_mediane_annualisee',r25,'contract_growth')
add_definition('contractuel_moyenne_annualisee',r25,'contract_growth','mean')
r25['effective_unannualized']=(r25.sRentEffective/r25.prev_sRentEffective-1)*100
add_definition('effectif_mediane_non_annualisee',r25,'effective_unannualized',meaning='périodes non normalisées')
no_concession=r25[r25.sRentEffective.eq(r25.sRent)&r25.prev_sRentEffective.eq(r25.prev_sRent)]
add_definition('sans_concession_dans_les_deux_baux',no_concession,'effective_growth',meaning='population sélectionnée')
for label,low,high in [('ecarts_0.9_1.1',.9,1.1),('ecarts_0.75_1.5',.75,1.5)]:
    pp,_=same_unit_pairs(known_2025,replace(CONFIG,min_gap_years=low,max_gap_years=high))
    add_definition(label,pp[pp.year.eq(2025)],'effective_growth',meaning='population sélectionnée par intervalle')
# Aucune jointure de concessions, aucune réimputation.
a=asking[asking.month_end.le(pd.Timestamp('2025-12-31'))].copy()
a['month_number']=a.date.dt.year*12+a.date.dt.month
previous=a[UNIT_KEY+['month_number','sAskingRent','date']].rename(columns={'sAskingRent':'previous_asking','date':'previous_date'})
previous['month_number']+=12
asking_pairs=a.merge(previous,on=UNIT_KEY+['month_number'],how='inner',validate='one_to_one')
asking_pairs=asking_pairs[asking_pairs.sAskingRent.gt(0)&asking_pairs.previous_asking.gt(0)].copy()
asking_pairs['asking_growth']=((asking_pairs.sAskingRent/asking_pairs.previous_asking)**(CONFIG.days_per_year/(asking_pairs.date-asking_pairs.previous_date).dt.days)-1)*100
asking_2025=asking_pairs[asking_pairs.date.dt.year.eq(2025)]
add_definition('annonces_mediane_meme_mois_12_mois',asking_2025,'asking_growth',meaning='panel annonces incomplet; transitions mensuelles')
add_definition('annonces_moyenne_meme_mois_12_mois',asking_2025,'asking_growth','mean',meaning='panel annonces incomplet; transitions mensuelles')
definition_sensitivity=pd.DataFrame(definition_rows)
display(definition_sensitivity[['definition','paires','unites','valeur_pct','statut']].round(3))
# Backtest de la règle A seulement pour chaque cible clairement nommée.
definition_backtest_rows=[]
for year in range(2019,2026):
    f=forecast_engine(leases,asking,external,year,'A_derniere')
    hist=f['samples']
    target=robust_pairs[robust_pairs.year.eq(year)]
    for label,column,agg in [('effectif_mediane','effective_growth','median'),('effectif_moyenne','effective_growth','mean'),
                             ('contractuel_mediane','contract_growth','median')]:
        pred=float(getattr(hist[column],agg)());actual=float(getattr(target[column],agg)())
        definition_backtest_rows.append({'definition':label,'year':year,'prediction_pct':pred,'observation_pct':actual,
                                        'erreur_absolue_pp':abs(pred-actual),'unites_reference':int(hist.hUnit.nunique()),
                                        'unites_observees':int(target.hUnit.nunique())})
definition_backtests=pd.DataFrame(definition_backtest_rows)
definition_backtests['periode']=np.where(definition_backtests.year.le(2022),'2019_2022','2023_2025')
definition_scores=definition_backtests.groupby(['definition','periode']).erreur_absolue_pp.agg(MAE_pp='mean',annees='size')
display(definition_scores.round(3))
print('Les scores de cibles différentes ne constituent pas une sélection de définition.')

## O — Disponibilité temporelle et maturité : limites vérifiables

Le backtest principal filtre les dates de signature et de début à fin d'année précédente avant toute construction des paires. Cela empêche l'accès informatique aux baux futurs; cela ne prouve pas la disponibilité réelle des effectifs ni l'absence de corrections tardives, faute de date de saisie. Tous les scores sont exploratoires.

La sensibilité stricte ci-dessous exige que les contrats de l'historique soient terminés à la coupure. L'évaluation garde la même cible principale. Une année n'est scorée que si les méthodes comparées et la cible atteignent le seuil de couverture configuré; sinon le notebook affiche la couverture et le motif, pas un score instable. Les observations cibles futures sont utilisées pour l'évaluation après prévision, jamais pour sélectionner les entrées du moteur.

Les sources externes sont filtrées selon leur date de publication. L'IPC emploie un millésime actuel, ce qui limite l'interprétation historique.

Repli préannoncé : si la dernière année partielle est trop petite, prendre la dernière année d'historique achevé suffisamment couverte. Les méthodes reçoivent la même année de référence et les mêmes données; les années partielles plus récentes sont exclues. Ce diagnostic change l'hypothèse de disponibilité et peut introduire un retard; il ne sélectionne pas une nouvelle méthode centrale. La date de saisie et le traitement définitif des crédits restent inconnus.

In [ ]:
strict_coverage_rows=[];strict_score_rows=[];maturity_rows=[]
for year in range(2019,2026):
    actual=robust_pairs[robust_pairs.year.eq(year)]
    mature=actual[actual.sLeaseTo.le(pd.Timestamp(year,12,31))]
    maturity_rows.append({'year':year,'unites_cible':int(actual.hUnit.nunique()),
                         'unites_baux_acheves_fin_annee':int(mature.hUnit.nunique()),
                         'paires_baux_acheves_fin_annee':len(mature)})
    forecasts=[]
    for method in METHODS:
        try:
            f=forecast_engine(leases,asking,external,year,method,replace(CONFIG,availability='conservative'))
        except ValueError:
            f=None
        forecasts.append(f)
    available=all(f is not None for f in forecasts)
    ref_units=min(f['reference_units'] for f in forecasts) if available else 0
    usable=available and ref_units>=CONFIG.min_output_pairs and actual.hUnit.nunique()>=CONFIG.min_output_pairs
    strict_coverage_rows.append({'year':year,'unites_reference_min':ref_units,'unites_cible':int(actual.hUnit.nunique()),
                                 'statut':'score_exploratoire' if usable else 'couverture_insuffisante'})
    if usable:
        for f in forecasts:
            observed=float(actual.effective_growth.median())
            strict_score_rows.append({'year':year,'method':f['method'],'prediction_pct':f['estimate_pct'],
                                      'observation_pct':observed,'erreur_absolue_pp':abs(f['estimate_pct']-observed),
                                      'unites_reference':f['reference_units']})
strict_coverage=pd.DataFrame(strict_coverage_rows)
strict_scores=pd.DataFrame(strict_score_rows)
target_maturity=pd.DataFrame(maturity_rows)
display(strict_coverage);display(target_maturity)
if not strict_scores.empty:
    display(strict_scores.round(3))
    display(strict_scores.groupby('method').erreur_absolue_pp.agg(MAE_pp='mean',annees='size').round(3))
else:
    print('Aucune année ne satisfait les seuils de couverture stricts; aucun score de maturité robuste annoncé.')
print('Disponibilité de saisie non démontrable; estimation centrale inchangée.')

# Repli de maturité annoncé : dernière année connue suffisamment couverte, sans changer la cible ni la sélection principale.
mature_lagged_rows=[];mature_lagged_coverage=[];same_window_principal_rows=[]
for year in range(2019,2026):
    cutoff=pd.Timestamp(year-1,12,31)
    completed=history_at(leases,cutoff,replace(CONFIG,availability='conservative'))
    hp,_=same_unit_pairs(completed)
    annual_units=hp.groupby('year').hUnit.nunique()
    eligible_years=annual_units[annual_units.ge(CONFIG.min_output_pairs)]
    if eligible_years.empty:
        mature_lagged_coverage.append({'year':year,'reference_year':None,'unites_reference':0,'statut':'aucune_annee_couverte'})
        continue
    ref_year=int(eligible_years.index.max())
    ref_units=int(eligible_years.loc[ref_year])
    mature_lagged_coverage.append({'year':year,'reference_year':ref_year,'unites_reference':ref_units,'statut':'score_exploratoire'})
    # Écarter les années partielles plus récentes avant d'appeler le moteur commun.
    restricted=leases[leases.sLeaseFrom.dt.year.le(ref_year)]
    actual=observed_target(leases,year)
    for method in METHODS:
        f=forecast_engine(restricted,asking,external,year,method,replace(CONFIG,availability='conservative'))
        assert f['reference_year']==ref_year and f['reference_units']>=CONFIG.min_output_pairs
        mature_lagged_rows.append({'year':year,'method':method,'reference_year':ref_year,'unites_reference':f['reference_units'],
                                  'prediction_pct':f['estimate_pct'],'observation_pct':actual['observed_pct'],
                                  'erreur_absolue_pp':abs(f['estimate_pct']-actual['observed_pct'])})
        principal=evaluate_method(leases,year,method)
        same_window_principal_rows.append({'year':year,'method':method,'erreur_absolue_pp':principal['absolute_error_pp']})
mature_lagged=pd.DataFrame(mature_lagged_rows)
mature_lagged_coverage=pd.DataFrame(mature_lagged_coverage)
same_window_principal=pd.DataFrame(same_window_principal_rows)
display(mature_lagged_coverage)
display(mature_lagged[['year','method','reference_year','unites_reference','prediction_pct','observation_pct','erreur_absolue_pp']].round(3))
matched_availability_scores=pd.concat([mature_lagged.assign(disponibilite='baux_termines_annee_couverte'),
                                      same_window_principal.assign(disponibilite='hypothese_contractuelle')]).groupby(['disponibilite','method']).erreur_absolue_pp.agg(MAE_pp='mean',annees='size')
display(matched_availability_scores.round(3))
assert mature_lagged.groupby('year').reference_year.nunique().eq(1).all()
assert mature_lagged.groupby('year').unites_reference.nunique().eq(1).all()
print('Diagnostic de maturité avec repli : jamais utilisé pour modifier la sélection centrale.')

completed_2025=history_at(leases,pd.Timestamp('2025-12-31'),replace(CONFIG,availability='conservative'))
hm,_=same_unit_pairs(completed_2025)
covered_years=hm.groupby('year').hUnit.nunique()
covered_years=covered_years[covered_years.ge(CONFIG.min_output_pairs)]
ref_2026=int(covered_years.index.max())
mature_2026_rows=[]
for method in METHODS:
    f=forecast_engine(leases[leases.sLeaseFrom.dt.year.le(ref_2026)],asking,external,2026,method,replace(CONFIG,availability='conservative'))
    mature_2026_rows.append({'method':method,'reference_year':f['reference_year'],'unites_reference':f['reference_units'],
                              'estimation_conditionnelle_pct':f['estimate_pct']})
mature_2026_scenarios=pd.DataFrame(mature_2026_rows)
display(mature_2026_scenarios.round(3))
print('Les trois variantes de maturité sont des scénarios, sans promotion de méthode centrale.')

## P — Contexte SCHL par marché : comparaison descriptive, pas une cible JADCO

Les tables officielles SCHL 2025 décrivent le parc locatif primaire d’immeubles privés de 3 logements et plus. La publication date du **11 décembre 2025**; les colonnes Octobre 2024 et Octobre 2025 viennent du même millésime publié en 2025. Elles ne constituent donc pas deux millésimes archivés disponibles en décembre 2024.

La cible interne JADCO demeure la médiane annuelle des transitions effectives comparables. Les statistiques SCHL sont des moyennes pour d’autres géographies et une autre population : elles donnent un contexte, jamais des baux comparables ou une valeur à substituer. Les appartements en copropriété loués sont couverts dans un sondage distinct dans certains centres. Les données ne représentent pas toutes les annonces.

Le tableau conserve les indicateurs de qualité SCHL (`a` à `d`) fournis. Les valeurs supprimées (`**`) et non calculées (`++`) ne sont pas converties en chiffres. Les valeurs d’inoccupation, loyer moyen, variation de loyer du même échantillon, rotation et parc proviennent respectivement des tables 1.1.1, 1.1.2, 1.1.5, 1.1.6 et 1.1.3. Le pourcentage de hausse à échantillon constant n’est pas le changement du loyer moyen du parc.

Licence consultée : [entente SCHL sur l’utilisation des données](https://www.cmhc-schl.gc.ca/professionals/housing-markets-data-and-research/housing-data/cmhc-licence-agreement-use-of-data). Les extraits et ce produit adapté doivent conserver l’attribution prescrite, ne pas suggérer d’appui SCHL, et demeurer sous les conditions de redistribution de l’entente. Les marques et logos SCHL ne sont pas repris.


**Passerelle deux chambres.** La comparaison rapproche la typologie, sans prétendre égaliser les cibles : les sites québécois observés sont Mont-Royal, Laval et Pointe-Claire; le site ontarien est Ottawa. Ces ensembles sont rapprochés des marchés SCHL Montréal et Ottawa (partie Ontario), dont le parc dépasse largement les six immeubles JADCO. Les taux internes sont des médianes de transitions annualisées commencées en 2025; la SCHL mesure une variation de loyer à échantillon constant entre enquêtes d'octobre. Les écarts numériques sont descriptifs : ni erreur du modèle, ni part causale des concessions. La SCHL n'entre pas dans la prévision A. Les qualités SCHL sont attachées séparément à chaque année, pour éviter de créer de faux trous lorsque la lettre de qualité change.


In [ ]:
schl = pd.read_csv(ROOT / 'externes' / 'schl-contexte-2025.csv')
schl = schl.dropna(subset=['date_publication']).copy()
assert set(schl.date_publication) == {'2025-12-11'}
assert schl.reference_period.eq(schl.annee_reference_octobre.astype(int).astype(str)+'-10').all()
assert schl.source_url.eq(schl.url_source).all()
assert schl.vintage.eq(schl.millesime).all()
assert set(schl.qualite_schl.dropna().unique()).issubset({'a','b','c','d'})
schl['annee_reference_octobre']=schl.annee_reference_octobre.astype(int)
schl_index=['geographie','mesure','unite']
assert not schl.duplicated(schl_index+['annee_reference_octobre']).any()
schl_values=schl.pivot(index=schl_index,columns='annee_reference_octobre',values='valeur')
schl_quality=schl.pivot(index=schl_index,columns='annee_reference_octobre',values='qualite_schl')
schl_values.columns=[f'valeur_{int(year)}' for year in schl_values.columns]
schl_quality.columns=[f'qualite_{int(year)}' for year in schl_quality.columns]
schl_market=schl_values.join(schl_quality,validate='one_to_one').reset_index()
display(schl_market)
print('Adapté de Canada Mortgage and Housing Corporation, Rental Market Survey Data Tables, référence octobre 2025. Cela ne constitue pas un endossement de ce produit par la SCHL.')


bridge_rows=[]
market_map={'Quebec':'Montréal CMA','Ontario':'Ottawa-Gatineau CMA (Ont. part)'}
city_map={'Quebec':{'Mont-Royal','Laval','Pointe-Claire'},'Ontario':{'Ottawa'}}
two_bed=r25[r25.sBeds.eq(2)]
for state,market in market_map.items():
    internal=two_bed[two_bed.sState.eq(state)]
    assert set(internal.sCity.dropna()).issubset(city_map[state]), 'Géographie interne à réexaminer'
    if internal.hUnit.nunique()<CONFIG.min_output_pairs:
        continue
    public=schl[schl.geographie.eq(market)&schl.mesure.eq('variation_loyer_meme_echantillon_2ch')&schl.annee_reference_octobre.eq(2025)]
    assert len(public)==1
    public_row=public.iloc[0]
    contractual=float(internal.contract_growth.median())
    effective=float(internal.effective_growth.median())
    public_growth=float(public_row.valeur)
    bridge_rows.append({'province':state,'marche_schl':market,'typologie':'2 chambres',
        'paires_internes':len(internal),'unites_internes':int(internal.hUnit.nunique()),
        'contractuel_mediane_annualisee_pct':contractual,'effectif_mediane_annualisee_pct':effective,
        'schl_meme_echantillon_pct':public_growth,'qualite_schl_2025':public_row.qualite_schl,
        'ecart_descriptif_contractuel_schl_pp':contractual-public_growth,
        'ecart_descriptif_effectif_schl_pp':effective-public_growth,
        'publication_schl':'2025-12-11','periode_interne':'transitions commencées en 2025',
        'periode_schl':'enquêtes octobre 2024 et octobre 2025',
        'comparabilite':'typologie rapprochée; parc, statistique, prix et calendrier différents; contexte seulement'})
external_bridge=pd.DataFrame(bridge_rows)
display(external_bridge[['province','paires_internes','unites_internes','contractuel_mediane_annualisee_pct',
                         'effectif_mediane_annualisee_pct','schl_meme_echantillon_pct','qualite_schl_2025']].round(3))
print('La proximité du contractuel québécois avec la SCHL ne valide pas la prévision effective 2026. Les écarts ne sont pas une décomposition causale.')

## Q — Archives SCHL : calendrier réel et ablation préannoncée

Huit classeurs officiels Montréal/Ottawa, éditions 2021–2024 : 160 observations, chacune avec cellule, qualité, URL et SHA-256. Le test n'utilise que les observations originales de chaque édition, deux chambres, hausse à échantillon constant. Québec → Montréal RMR; Ontario → Ottawa partie Ontario. Les suppressions ne sont jamais imputées.

Le protocole `PROTOCOLE-NOUVELLES-DONNEES.md` a été écrit avant les scores : une seule variante D, même translation segmentée que B, ancrage fixe 50 % vers la SCHL, mêmes années/coupures/populations. Comparaison MAE et erreurs annuelles; seuil pratique inchangé de gain ≥0,25 point et ≥2/3 années améliorées. Aucun ajustement ultérieur de poids. La statistique externe reste une moyenne contractuelle du parc, différente de notre médiane effective de transitions.

L'édition 2021 est publiée le 18 février 2022, 2022 le 26 janvier 2023, 2023 le 31 janvier 2024 et 2024 le 17 décembre 2024. Ainsi l'enquête de l'année précédente n'est pas systématiquement disponible : l'âge du signal est d'environ quatorze mois pour les deux premières coupures, contre deux mois pour la dernière. L'édition 2025 est disponible pour prévoir 2026, mais aucune promotion de D ne découle de ce seul fait.

Les fichiers historiques ont été téléchargés en 2026 : leurs millésimes sont documentés, sans preuve absolue d'absence de correction silencieuse. Les années 2023–2025 restent exploratoires. La table finale illustre le passage des géographies de recensement 2016 à 2021 : pour Ottawa 2022, une révision présentée dans l’édition 2023, publiée le 31 janvier 2024 n'est pas une croissance et ne doit pas remplacer la valeur connue auparavant. Voir `SOURCES-ET-LICENCES.md` et le registre de provenance.

In [ ]:
def select_schl_asof(source, cutoff, required_states):
    """Dernière édition originale publiée; aucune imputation ni repli d'une suppression."""
    mapping = {'Quebec': 'Montréal RMR',
               'Ontario': 'Ottawa-Gatineau RMR, partie Ontario'}
    cutoff = pd.Timestamp(cutoff)
    data = source.copy()
    data['publication'] = pd.to_datetime(data.publication, errors='raise')
    data['reference_date'] = pd.to_datetime(data.periode_reference + '-01', errors='raise')
    data = data[data.edition.eq(data.annee_reference)
                & data.chambres.astype(str).eq('2')
                & data.indicateur.eq('hausse_meme_echantillon_pct')
                & data.publication.le(cutoff) & data.reference_date.le(cutoff)]
    rows = []
    for state in sorted(set(required_states)):
        if state not in mapping:
            raise ValueError('Province sans correspondance SCHL vérifiée')
        candidates = data[data.geographie.eq(mapping[state])]
        if candidates.empty:
            raise ValueError('Marché SCHL manquant à la coupure')
        newest = candidates[candidates.publication.eq(candidates.publication.max())]
        if len(newest) != 1:
            raise ValueError('Édition SCHL ambiguë ou dupliquée')
        row = newest.iloc[0].copy()
        value = pd.to_numeric(row.valeur, errors='coerce')
        if row.statut != 'publie' or not np.isfinite(value) or row.qualite not in ('a', 'b', 'c', 'd'):
            raise ValueError('Signal SCHL supprimé, manquant ou qualité inconnue')
        row['geography'] = state
        row['value'] = float(value)
        rows.append(row)
    return pd.DataFrame(rows).reset_index(drop=True)


def forecast_schl_anchor(frame, asking_frame, source, year, config=CONFIG):
    """Ablation D: même translation segmentée que B, ancre SCHL fixe 50 %."""
    base = forecast_engine(frame, asking_frame, None, year, 'A_derniere', config)
    reference = base['samples'].copy()
    signals = select_schl_asof(source, base['cutoff'], reference.sState.unique())
    hist, _ = same_unit_pairs(history_at(frame, pd.Timestamp(base['cutoff']), config), config)
    predicted = reference['predicted_growth'].to_numpy().copy()
    for (state, renewal), group in reference.groupby(['sState', 'sRenewal']):
        location, _ = historical_location(hist, state, renewal, config, 'effective_growth')
        anchor = float(signals.loc[signals.geography.eq(state), 'value'].iloc[0])
        location = 0.5 * location + 0.5 * anchor
        positions = reference.index.get_indexer(group.index)
        predicted[positions] += location - float(group.effective_growth.median())
    reference['predicted_growth'] = predicted
    base.update(method='D_B_plus_SCHL_50', estimate_pct=float(np.median(predicted)),
                external_rows_available=len(signals), samples=reference, signals=signals)
    return base


schl_history = pd.read_csv(PUBLIC_DIR / 'schl-historique-2021-2024.csv')
schl_rows, ledger_rows = [], []
for year in TEST_YEARS:
    candidate = forecast_schl_anchor(leases, asking, schl_history, year)
    actual = observed_target(leases, year)
    row = {k: v for k, v in candidate.items() if k not in ('samples', 'signals')}
    row.update(actual)
    row['error_pp'] = row['estimate_pct'] - row['observed_pct']
    row['absolute_error_pp'] = abs(row['error_pp'])
    schl_rows.append(row)
    for signal in candidate['signals'].itertuples():
        ledger_rows.append({'prevision': year, 'province': signal.geography,
            'edition': int(signal.edition), 'reference': signal.periode_reference,
            'publication': str(signal.publication.date()), 'hausse_pct': signal.value,
            'qualite': signal.qualite})
schl_ablation = pd.DataFrame(schl_rows)
schl_ledger = pd.DataFrame(ledger_rows)
schl_errors = schl_ablation.set_index('year').absolute_error_pp
schl_verdict = {'method': 'D_B_plus_SCHL_50', 'weight': 0.5,
    'MAE_pp': float(schl_errors.mean()), 'erreur_max_pp': float(schl_errors.max()),
    'gain_vs_A_pp': float(baseline_errors.mean() - schl_errors.mean()),
    'gain_vs_C_pp': float(summary.loc['C_B_plus_IPC', 'MAE_pp'] - schl_errors.mean()),
    'annees_ameliorees_vs_A': int(schl_errors.lt(baseline_errors).sum()),
    'passes_numerical_gate': bool(baseline_errors.mean()-schl_errors.mean() >= CONFIG.complexity_gain_pp
                                  and schl_errors.lt(baseline_errors).sum() >= 2),
    'independent_validation': False,
    'population_comparable': False,
    'meaning': 'Ancre moyenne contractuelle deux chambres versus médiane effective de transitions; pas de promotion automatique.'}
display(schl_ledger)
display(schl_ablation[['year', 'estimate_pct', 'observed_pct', 'error_pp',
                       'absolute_error_pp', 'reference_pairs', 'observed_pairs']].round(3))
schl_comparison = summary[['MAE_pp', 'erreur_max_pp']].copy()
schl_comparison.loc['D_B_plus_SCHL_50'] = [schl_verdict['MAE_pp'], schl_verdict['erreur_max_pp']]
display(schl_comparison.round(3))
display(Markdown(f"**Résultat de l'ablation annoncée :** MAE {schl_verdict['MAE_pp']:.3f} point; "
    f"gain contre A {schl_verdict['gain_vs_A_pp']:+.3f} point et "
    f"{schl_verdict['annees_ameliorees_vs_A']}/3 années améliorées. "
    f"Seuil numérique franchi : {schl_verdict['passes_numerical_gate']}. "
    "Les archives renforcent la traçabilité; elles ne justifient pas à elles seules un changement de prévision. "
    "Aucun poids supplémentaire n'a été recherché après consultation des scores."))

# Même référence 2022, deux éditions / périmètres : ne pas confondre révision et croissance.
schl_revisions = schl_history[
    schl_history.geographie.eq('Ottawa-Gatineau RMR, partie Ontario')
    & schl_history.annee_reference.eq(2022) & schl_history.chambres.astype(str).eq('2')
    & schl_history.indicateur.isin(['univers_unites', 'loyer_moyen_cad'])
][['edition', 'geographie_recensement', 'indicateur', 'valeur', 'publication']].sort_values(['indicateur', 'edition'])
display(schl_revisions)

## R — Offre locative en construction : surveiller la concurrence

Rapport SCHL automne 2025, publié le **9 septembre 2025**, disponible avant la coupure 2026. La figure 8 décrit des logements spécifiquement locatifs en construction à Montréal/Laval. Les labels 2024/2025 ne prouvent pas un mois historique précis; le titre mentionne la mi-2025.

**Stock en construction ≠ livraisons**, ni calendrier d'achèvement, ni causalité estimée sur nos loyers. Ce millésime 2025 ne rentre pas dans le backtest 2023–2025. Ottawa est exclu de ce rapprochement : la figure correspondante mélange les marchés visés et ne confirme pas ici la portion Ontario. Ces nouvelles données servent à définir une surveillance prospective, sans coefficient arbitraire ajouté à la prévision.

In [ ]:
offer_source = pd.read_csv(PUBLIC_DIR / 'schl-offre-stock-2025.csv', dtype={'reference_period': str})
offer_selected = offer_source[
    offer_source.geography.eq('Montreal CMA')
    & offer_source.dimension.isin(['Island of Montreal', 'Laval'])
    & offer_source.reference_period.isin(['2024', '2025'])
    & offer_source.intended_market.eq('purpose_built_rental')
    & offer_source.metric.eq('units_under_construction')
    & pd.to_datetime(offer_source.publication_date).le(pd.Timestamp('2025-12-31'))
].copy()
assert len(offer_selected) == 4
offer_context = offer_selected.pivot(index='dimension', columns='reference_period', values='value')
offer_context['variation_stock_pct'] = (offer_context['2025']/offer_context['2024']-1)*100
offer_context = offer_context.rename_axis('zone').reset_index()
display(offer_context.round(2))
display(Markdown("**Usage décisionnel :** suivre chaque mois, par immeuble et taille, les délais "
    "entre disponibilité et signature, les concessions et le loyer effectif des nouvelles locations. "
    "Conserver les logements encore disponibles dans le dénominateur et traiter leur délai comme censuré; "
    "ne pas mesurer seulement les logements déjà loués. Comparer à la même saison, afficher effectifs "
    "et couverture, puis examiner les cas avec le responsable locatif. "
    "Une hausse des concessions accompagnée d'un allongement des délais déclenche une revue du scénario "
    "de relocation, sans imputer automatiquement cet effet aux constructions. "
    "Seuils d'alerte et horizon à fixer prospectivement avec JADCO; aucun seuil validé n'est revendiqué."))

## S — Dans quels segments la méthode est-elle évaluable?

Diagnostic exploratoire annoncé avant le calcul des scores locaux : méthode A figée, années de test configurées, même cible effective et mêmes coupures. Chaque groupe doit atteindre le seuil d'unités distinctes en référence et en cible. Les groupes nouveaux, disparus ou inconnus restent visibles avec leur motif d'abstention.

Une erreur locale sur une seule année ne permet pas de classer ce groupe face à un groupe couvert sur toute la période. Ce diagnostic n'améliore pas rétrospectivement la méthode et ne prouve pas sa précision future. Les dates de saisie de l'effectif restent inconnues.

In [ ]:
def _segment_units(frame):
    return int(len(frame[UNIT_KEY].drop_duplicates()))

def _segment_groups(frame, keys):
    result = {}
    for key, group in frame.groupby(keys, dropna=False, observed=True, sort=False):
        if not isinstance(key, tuple):
            key = (key,)
        normalized = tuple(None if pd.isna(value) else value for value in key)
        result[normalized] = group
    return result

def _segment_label(keys, key):
    return ' | '.join(f'{name}={"inconnu" if value is None else value}'
                      for name, value in zip(keys, key))

def segment_year_diagnostics(frame, asking_frame, external_frame, year, axes, config=CONFIG):
    cutoff = pd.Timestamp(year - 1, 12, 31)
    # Empty history is an explicit abstention, not an exception masking other failures.
    eligible, _ = same_unit_pairs(history_at(frame, cutoff, config), config)
    if eligible.empty:
        reference = eligible.copy()
        reference['predicted_growth'] = pd.Series(dtype=float)
        reference_year = None
    else:
        forecast = forecast_engine(frame, asking_frame, external_frame, year,
                                   'A_derniere', config)
        reference = forecast['samples']
        reference_year = forecast['reference_year']
    # Targets never enter the forecast engine. Pair construction precedes segmentation.
    data = prepare_leases(frame)
    observed, _ = same_unit_pairs(data[data.sLeaseFrom.le(pd.Timestamp(year, 12, 31))], config)
    target = observed[observed.year.eq(year)]
    rows, coverage = [], []
    for axis, keys in axes:
        reference_groups = _segment_groups(reference, keys)
        target_groups = _segment_groups(target, keys)
        union_keys = sorted(set(reference_groups) | set(target_groups),
                            key=lambda key: tuple((value is not None, str(value)) for value in key))
        scored_reference, scored_target = [], []
        for key in union_keys:
            ref = reference_groups.get(key, reference.iloc[:0])
            obs = target_groups.get(key, target.iloc[:0])
            ref_units, obs_units = _segment_units(ref), _segment_units(obs)
            if any(value is None for value in key):
                status = 'groupe_inconnu'
            elif ref.empty:
                status = 'absent_reference'
            elif obs.empty:
                status = 'absent_cible'
            elif min(ref_units, obs_units) < config.min_output_pairs:
                status = 'couverture_insuffisante'
            else:
                status = 'calculable'
            row = {'axis': axis, 'segment': _segment_label(keys, key), 'year': int(year),
                   'cutoff': str(cutoff.date()), 'reference_year': reference_year,
                   'reference_pairs': len(ref), 'reference_units': ref_units,
                   'observed_pairs': len(obs), 'observed_units': obs_units, 'status': status,
                   'estimate_pct': None, 'observed_pct': None,
                   'error_pp': None, 'absolute_error_pp': None}
            if status == 'calculable':
                estimate = float(ref.predicted_growth.median())
                actual = float(obs.effective_growth.median())
                row.update(estimate_pct=estimate, observed_pct=actual,
                           error_pp=estimate-actual, absolute_error_pp=abs(estimate-actual))
                scored_reference.append(ref)
                scored_target.append(obs)
            rows.append(row)
        cover = {'axis': axis, 'year': int(year), 'groups_total': len(union_keys),
                 'groups_scored': len(scored_reference)}
        for prefix, all_rows, scored in [('reference', reference, scored_reference),
                                         ('observed', target, scored_target)]:
            selected = pd.concat(scored) if scored else all_rows.iloc[:0]
            pairs_total, units_total = len(all_rows), _segment_units(all_rows)
            pairs_scored, units_scored = len(selected), _segment_units(selected)
            cover.update({prefix+'_pairs_total': pairs_total, prefix+'_units_total': units_total,
                          prefix+'_pairs_scored': pairs_scored, prefix+'_units_scored': units_scored,
                          prefix+'_pair_coverage_pct': 100*pairs_scored/pairs_total if pairs_total else None,
                          prefix+'_unit_coverage_pct': 100*units_scored/units_total if units_total else None})
        coverage.append(cover)
    return rows, coverage

def summarize_segment_diagnostics(rows, years):
    expected = sorted(set(int(year) for year in years))
    groups = {}
    for row in rows:
        if row['year'] in expected:
            groups.setdefault((row['axis'], row['segment']), []).append(row)
    summary_rows = []
    for (axis, segment), group in sorted(groups.items()):
        scored = sorted((row for row in group if row['status'] == 'calculable'),
                        key=lambda row: row['year'])
        scored_years = [row['year'] for row in scored]
        if len(set(scored_years)) != len(scored_years):
            raise ValueError('Année segment dupliquée : synthèse ambiguë')
        errors = [row['error_pp'] for row in scored]
        status = ('comparable_3_sur_3' if scored_years == expected and len(expected) == 3
                  else 'incomplet' if scored else 'non_calculable')
        summary_rows.append({'axis': axis, 'segment': segment, 'years_scored': scored_years,
                             'years_expected': len(expected), 'n_years': len(scored), 'status': status,
                             'MAE_pp': float(np.mean(np.abs(errors))) if errors else None,
                             'max_error_pp': float(np.max(np.abs(errors))) if errors else None,
                             'mean_error_pp': float(np.mean(errors)) if errors else None})
    return summary_rows

segment_axes = [('type', ['sRenewal']), ('province_type', ['sState', 'sRenewal']),
                ('immeuble', ['sBuilding']), ('province_chambres', ['sState', 'sBeds'])]
segment_rows, segment_coverage = [], []
for segment_year in TEST_YEARS:
    year_rows, year_coverage = segment_year_diagnostics(leases, asking, external,
                                                     segment_year, segment_axes)
    segment_rows.extend(year_rows)
    segment_coverage.extend(year_coverage)
segment_summary = summarize_segment_diagnostics(segment_rows, TEST_YEARS)
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(pd.DataFrame(segment_summary)[['axis', 'segment', 'years_scored', 'status',
                                          'MAE_pp', 'max_error_pp', 'mean_error_pp']].round(3))
    display(pd.DataFrame(segment_coverage)[['axis', 'year', 'groups_scored', 'groups_total',
                                           'reference_unit_coverage_pct', 'observed_unit_coverage_pct']].round(2))
    annual_segments = pd.DataFrame(segment_rows)
    display(annual_segments[annual_segments.axis.isin(['type', 'province_type'])][
        ['axis', 'segment', 'year', 'reference_units', 'observed_units', 'status',
         'estimate_pct', 'observed_pct', 'error_pp']].round(3))
display(Markdown('Les sorties insuffisamment couvertes restent vides. Les erreurs de groupes '
                 'ne se moyennent pas pour retrouver la MAE globale. Aucun nouveau choix de modèle.'))

## L — Références et reproductibilité

- Consignes CodeML 2026, JADCO, et notebook de départ `starter.ipynb`. La définition est libre mais doit être défendue; Python/Jupyter et une documentation des sources, bibliothèques et outils d'IA sont demandés.
- [StatCan 18-10-0004-01](https://www150.statcan.gc.ca/t1/tbl1/fr/tv.action?pid=1810000401), composante loyer. Les sources publiques peuvent être fournies localement; leurs transformations sont dans la section G.
- [SCHL ELL](https://www.cmhc-schl.gc.ca/professionals/housing-markets-data-and-research/housing-data/data-tables/rental-market/rental-market-report-data-tables). Les données SCHL servent de contexte externe et les restrictions de licence doivent être respectées.
- [TAL, augmentation](https://www.tal.gouv.qc.ca/fr/reconduction-du-bail-et-fixation-de-loyer/augmentation-de-loyer) et [modification du bail](https://www.tal.gouv.qc.ca/fr/reconduction-du-bail-et-fixation-de-loyer/modification-d-une-condition-du-bail). Aucun plafond uniforme n'est déduit.
- [Ontario, hausses résidentielles](https://www.ontario.ca/page/residential-rent-increases). L'applicabilité et les exceptions doivent être vérifiées pour chaque bail.
- Outil IA utilisé : **OpenAI Codex**, assistance au code, à l'audit et à la rédaction. Aucun article scientifique ni poids préentraîné utilisé.

Les méthodes sont des règles statistiques calculées sur l'historique; aucun modèle entraîné n'est utilisé. Les fichiers de données du défi et les résultats calculés ne sont pas inclus dans cette copie publique.

### Exécution locale

Installer les dépendances de `requirements.txt`, puis ouvrir ce notebook dans Jupyter. Fournir localement les quatre CSV autorisés dans `entrees-locales/` ou définir `JADCO_INPUT_DIR`; aucune donnée CRM n'est incluse ici. Les sources publiques externes attendues par les sections correspondantes doivent également être disponibles sous `externes/`. Exécuter les cellules dans l'ordre. Les sorties apparaissent dans l'interface locale; ne pas les enregistrer dans la copie publique ni les committer.